# Independent check with PYPOWER

All other notebooks build the admittance matrix from the stored Yff/Yft/Ytf/Ytt entries of the released tables and solve
it with our Newton-Raphson or lightsim2grid, so they could share a modelling error. This notebook rebuilds the most
important cases from the raw network parameters (r, x, b, tap, shift, status, bus shunts, generator table) with PYPOWER,
the Python port of MATPOWER, and solves them with PYPOWER's own Newton-Raphson and reactive-limit loop:

1. conventions: units, bus types, taps, shunts, branch and generator status, the slack, and whether the released
   solutions satisfy the independently built equations;
2. ordinary cases and cases close to a voltage or thermal limit: same decisions?
3. constructed N-20 cases that no classical start solved;
4. representative folds from `storms`: same fold, same solution branch?
5. cases where reactive limits change the outcome, under the MATPOWER convention and ours;
6. the cases where our own methods disagreed.

Runtime: CPU, about 1 GB of downloads, 30 to 45 minutes. Output: `independent_check.zip`.

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "independent_check"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVEL_BASE, BASE_PARTS = 2, [0, 1]   # the two N-2 data blocks of the storm base points (as in the storm studies)
N_BASE = 120                         # base points, drawn exactly as in the storm studies (same seed)
N_CHECK = 24                         # base graphs on which the loaded model is checked (CPU)
N_ORDINARY, N_NEAR = 30, 20          # released N-2 cases validated: random, and nearest to a limit
TOTAL_OUT, REGION_HOPS, FOOT_HOPS, FOOT_SCALE, BETWEENNESS_SAMPLES = 20, 4, 6, 2.0, 400   # the construction constructions, replayed exactly
BATCH = 4
TOL, TOL_SCREEN, MAX_NR, SCREEN_MAX = 1e-8, 1e-2, 30, 6
BAD_VM = 0.01
MAX_DOWNLOAD_GB = 3
SEED = 0
PLAN_JSON = '{"fold_storms":[{"sid":910000002,"base":1,"removed_lines":[1590,1553,1591,1403,2088,1833,3170,1345,1831,1419,1346,1830,1567,1418,1282,1161,3216,2176,1570],"lam_max":0.55,"lam_fold":0.5502085147362109,"solvable_at_1":false,"weak_bus":439,"rising":true},{"sid":910000031,"base":109,"removed_lines":[1760,2131,1918,2132,1755,1753,1754,296,1741,295,1947,2030,1997,1995,2028,1751,1672,1998],"lam_max":0.6925,"lam_fold":0.6943711752823644,"solvable_at_1":false,"weak_bus":1275,"rising":true},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"rising":true},{"sid":910000081,"base":257,"removed_lines":[503,820,770,775,818,714,734,573,767,773,574,689,736,670,684,589,730,2099,1977],"lam_max":0.39875,"lam_fold":0.3992912982554781,"solvable_at_1":false,"weak_bus":439,"rising":true},{"sid":910000021,"base":55,"removed_lines":[456,458,261,256,435,488,433,160,489,385,877,48,378,383,123,369,393,436,1349],"lam_max":0.52375,"lam_fold":0.5243721581610346,"solvable_at_1":false,"weak_bus":88,"rising":false},{"sid":910000016,"base":32,"removed_lines":[253,199,175,165,136,275,283,282,128,955,3195,956,952,3194,957,3136,3048,134],"lam_max":0.5025,"lam_fold":0.5032461951655914,"solvable_at_1":false,"weak_bus":1945,"rising":false},{"sid":910000001,"base":0,"removed_lines":[1540,1537,1259,1542,945,1296,1544,1466,1256,1543,1541,1084,3096,1367,1031,944,925,1369,1366],"lam_max":1.645,"lam_fold":1.6469921856230798,"solvable_at_1":true,"weak_bus":626,"rising":false},{"sid":910000098,"base":322,"removed_lines":[1494,1495,1593,1493,1492,1491,1444,1085,1032,1232,1445,926,1099,1081,1506,1082,289,956],"lam_max":1.545,"lam_fold":1.545220200802817,"solvable_at_1":true,"weak_bus":497,"rising":false}],"qlim_storms":[{"sid":910000012,"base":24,"removed_lines":[640,639,807,691,784,642,644,643,782,792,684,2253,793,590,790,1691,563,589,692],"lam_max":1.0225,"lam_fold":1.024693791982196,"solvable_at_1":true,"weak_bus":539,"solvable_no_limits":true,"solvable_with_limits":false,"generators_at_a_limit":null},{"sid":910000019,"base":47,"removed_lines":[3088,3041,3040,3179,3063,3020,3067,3018,3086,3066,3019,3012,3187,3142,3167,2587,2586,2588],"lam_max":1.04,"lam_fold":1.0416041380984222,"solvable_at_1":true,"weak_bus":1964,"solvable_no_limits":true,"solvable_with_limits":false,"generators_at_a_limit":null},{"sid":910000041,"base":134,"removed_lines":[42,13,45,44,412,63,40,101,38,334,39,52,37,379,87,380,403,60,405],"lam_max":1.1,"lam_fold":1.1018728011671155,"solvable_at_1":true,"weak_bus":26,"solvable_no_limits":true,"solvable_with_limits":false,"generators_at_a_limit":null},{"sid":910000052,"base":169,"removed_lines":[17,57,59,95,105,98,55,61,22,115,396,395,119,20,499,526,407,38],"lam_max":1.0225,"lam_fold":1.024011381423925,"solvable_at_1":true,"weak_bus":45,"solvable_no_limits":true,"solvable_with_limits":false,"generators_at_a_limit":null},{"sid":910000042,"base":136,"removed_lines":[3089,1258,3097,1257,945,1259,1548,3127,1256,946,3208,3130,3115,3113,2646,925,3100,947],"lam_max":1.16,"lam_fold":1.1610286441811737,"solvable_at_1":true,"weak_bus":1850,"solvable_no_limits":true,"solvable_with_limits":true,"generators_at_a_limit":41},{"sid":910000083,"base":264,"removed_lines":[940,942,389,939,938,390,941,118,1385,872,1571,117,1136,384,946,393,116,879],"lam_max":1.4875,"lam_fold":1.489486370534525,"solvable_at_1":true,"weak_bus":685,"solvable_no_limits":true,"solvable_with_limits":true,"generators_at_a_limit":49},{"sid":910000090,"base":287,"removed_lines":[1450,1318,1449,1448,1072,1292,1321,1322,1359,1405,1291,1341,1171,1237,1306,1357,1530,1363,1425],"lam_max":1.6675,"lam_fold":1.66757554049419,"solvable_at_1":true,"weak_bus":253,"solvable_no_limits":true,"solvable_with_limits":true,"generators_at_a_limit":54},{"sid":910000003,"base":2,"removed_lines":[2025,2031,2028,1948,2030,2029,1635,1753,2033,2102,1996,1946,2027,1755,2032,2026,1634,1752,2133],"lam_max":1.4325,"lam_fold":1.433992161351087,"solvable_at_1":true,"weak_bus":1248,"solvable_no_limits":true,"solvable_with_limits":true,"generators_at_a_limit":16},{"sid":910000005,"base":6,"removed_lines":[1632,1637,1633,1634,1635,2030,2028,1975,1941,1942,2032,2033,1974,1675,1623,2000,1756,1626],"lam_max":1.26,"lam_fold":1.2612837948951583,"solvable_at_1":true,"weak_bus":1217,"solvable_no_limits":true,"solvable_with_limits":true,"generators_at_a_limit":17}],"failed_points":[{"sid":910000002,"base":1,"removed_lines":[1590,1553,1591,1403,2088,1833,3170,1345,1831,1419,1346,1830,1567,1418,1282,1161,3216,2176,1570],"lam_max":0.55,"lam_fold":0.5502085147362109,"solvable_at_1":false,"weak_bus":439,"lam":0.53,"lam_rel":-0.02,"flat_converged":true,"dc_converged":true,"true_min_vm":0.9019611512354764},{"sid":910000002,"base":1,"removed_lines":[1590,1553,1591,1403,2088,1833,3170,1345,1831,1419,1346,1830,1567,1418,1282,1161,3216,2176,1570],"lam_max":0.55,"lam_fold":0.5502085147362109,"solvable_at_1":false,"weak_bus":439,"lam":0.54,"lam_rel":-0.01,"flat_converged":false,"dc_converged":true,"true_min_vm":0.9402528653903788},{"sid":910000002,"base":1,"removed_lines":[1590,1553,1591,1403,2088,1833,3170,1345,1831,1419,1346,1830,1567,1418,1282,1161,3216,2176,1570],"lam_max":0.55,"lam_fold":0.5502085147362109,"solvable_at_1":false,"weak_bus":439,"lam":0.55,"lam_rel":0.0,"flat_converged":false,"dc_converged":true,"true_min_vm":0.950536915965173},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.77375,"lam_rel":-0.1499999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.5116991040109099},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.78375,"lam_rel":-0.14,"flat_converged":false,"dc_converged":false,"true_min_vm":0.5305226924907455},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.79375,"lam_rel":-0.13,"flat_converged":false,"dc_converged":false,"true_min_vm":0.5504654762609352},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.80375,"lam_rel":-0.12,"flat_converged":false,"dc_converged":false,"true_min_vm":0.5716371254083905},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.81375,"lam_rel":-0.1099999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.5923084249291625},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.82375,"lam_rel":-0.0999999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.6117395936150744},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.83375,"lam_rel":-0.0899999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.6326698788942482},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.84375,"lam_rel":-0.0799999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.6552952272342432},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.85375,"lam_rel":-0.0699999999999999,"flat_converged":false,"dc_converged":true,"true_min_vm":0.6752930577369893},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.86375,"lam_rel":-0.0599999999999999,"flat_converged":false,"dc_converged":false,"true_min_vm":0.6946887523946839},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.87375,"lam_rel":-0.0499999999999999,"flat_converged":false,"dc_converged":true,"true_min_vm":0.7164540141848184},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.88375,"lam_rel":-0.0399999999999999,"flat_converged":false,"dc_converged":true,"true_min_vm":0.7412006067097238},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.89375,"lam_rel":-0.0299999999999999,"flat_converged":false,"dc_converged":true,"true_min_vm":0.7656750273278234},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.90375,"lam_rel":-0.0199999999999999,"flat_converged":false,"dc_converged":true,"true_min_vm":0.7869790037934872},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.91375,"lam_rel":-0.01,"flat_converged":false,"dc_converged":true,"true_min_vm":0.8162666983347255},{"sid":910000043,"base":137,"removed_lines":[2587,2589,2590,2586,3019,2975,2712,2354,2997,3017,2280,3002,2351,2995,2709,3021,3020,3142],"lam_max":0.92375,"lam_fold":0.9240032087808512,"solvable_at_1":false,"weak_bus":1989,"lam":0.92375,"lam_rel":0.0,"flat_converged":false,"dc_converged":true,"true_min_vm":0.8705688663063996},{"sid":910000098,"base":322,"removed_lines":[1494,1495,1593,1493,1492,1491,1444,1085,1032,1232,1445,926,1099,1081,1506,1082,289,956],"lam_max":1.545,"lam_fold":1.545220200802817,"solvable_at_1":true,"weak_bus":497,"lam":1.395,"lam_rel":-0.1499999999999999,"flat_converged":true,"dc_converged":true,"true_min_vm":0.8301922049118219},{"sid":910000098,"base":322,"removed_lines":[1494,1495,1593,1493,1492,1491,1444,1085,1032,1232,1445,926,1099,1081,1506,1082,289,956],"lam_max":1.545,"lam_fold":1.545220200802817,"solvable_at_1":true,"weak_bus":497,"lam":1.405,"lam_rel":-0.1399999999999999,"flat_converged":true,"dc_converged":true,"true_min_vm":0.8268037078836781}],"n2_fingerprint":[[940000001,10,"nearest",3.0,true],[940000002,10,"uniform",15.0,true],[940000003,10,"corridor",4.0,true],[940000004,10,"footprint",6.0,true],[940000005,20,"nearest",1.0,true],[940000006,20,"uniform",16.0,true],[940000007,20,"corridor",4.0,true],[940000008,20,"footprint",6.0,true],[940000009,33,"nearest",2.0,true],[940000010,33,"uniform",16.0,true],[940000011,33,"corridor",4.0,true],[940000012,33,"footprint",6.0,true],[940000013,64,"nearest",7.0,false],[940000014,64,"uniform",15.0,true],[940000015,64,"corridor",5.0,false],[940000016,64,"footprint",6.0,false],[940000017,86,"nearest",2.0,true],[940000018,86,"uniform",16.0,true],[940000019,86,"corridor",5.0,true],[940000020,86,"footprint",5.0,true],[940000021,110,"nearest",3.0,true],[940000022,110,"uniform",22.0,true],[940000023,110,"corridor",4.0,true],[940000024,110,"footprint",6.0,true],[940000025,132,"nearest",2.0,true],[940000026,132,"uniform",16.0,true],[940000027,132,"corridor",4.0,true],[940000028,132,"footprint",6.0,true],[940000029,161,"nearest",3.0,false],[940000030,161,"uniform",17.0,true],[940000031,161,"corridor",4.0,false],[940000032,161,"footprint",6.0,false],[940000033,195,"nearest",3.0,true],[940000034,195,"uniform",19.0,true],[940000035,195,"corridor",4.0,true],[940000036,195,"footprint",6.0,true],[940000037,210,"nearest",3.0,false],[940000038,210,"uniform",15.0,true],[940000039,210,"corridor",4.0,true],[940000040,210,"footprint",5.0,true],[940000041,235,"nearest",1.0,true],[940000042,235,"uniform",18.0,true],[940000043,235,"corridor",4.0,true],[940000044,235,"footprint",6.0,true],[940000045,289,"nearest",6.0,false],[940000046,289,"uniform",18.0,true],[940000047,289,"corridor",4.0,false],[940000048,289,"footprint",6.0,false],[940000049,292,"nearest",5.0,true],[940000050,292,"uniform",16.0,true],[940000051,292,"corridor",4.0,true],[940000052,292,"footprint",6.0,true],[940000053,314,"nearest",4.0,true],[940000054,314,"uniform",16.0,true],[940000055,314,"corridor",4.0,true],[940000056,314,"footprint",6.0,true],[940000057,315,"nearest",5.0,true],[940000058,315,"uniform",25.0,true],[940000059,315,"corridor",4.0,true],[940000060,315,"footprint",6.0,true],[940000061,333,"nearest",3.0,true],[940000062,333,"uniform",18.0,true],[940000063,333,"corridor",5.0,true],[940000064,333,"footprint",6.0,true],[940000065,348,"nearest",4.0,true],[940000066,348,"uniform",19.0,true],[940000067,348,"corridor",4.0,true],[940000068,348,"footprint",6.0,true],[940000069,484,"nearest",2.0,true],[940000070,484,"uniform",20.0,true],[940000071,484,"corridor",6.0,true],[940000072,484,"footprint",6.0,true],[940000073,528,"nearest",2.0,true],[940000074,528,"uniform",19.0,true],[940000075,528,"corridor",4.0,true],[940000076,528,"footprint",6.0,true],[940000077,596,"nearest",3.0,true],[940000078,596,"uniform",24.0,true],[940000079,596,"corridor",5.0,false],[940000080,596,"footprint",6.0,true],[940000081,1278,"nearest",3.0,true],[940000082,1278,"uniform",20.0,true],[940000083,1278,"corridor",4.0,false],[940000084,1278,"footprint",6.0,true],[940000085,1285,"nearest",2.0,true],[940000086,1285,"uniform",16.0,true],[940000087,1285,"corridor",4.0,true],[940000088,1501,"nearest",2.0,true],[940000089,1501,"uniform",15.0,true],[940000090,1501,"corridor",4.0,false],[940000091,1501,"footprint",6.0,true],[940000092,1520,"nearest",3.0,true],[940000093,1520,"uniform",17.0,true],[940000094,1520,"corridor",4.0,false],[940000095,1520,"footprint",6.0,true],[940000096,1551,"nearest",7.0,true],[940000097,1551,"uniform",16.0,true],[940000098,1551,"corridor",4.0,true],[940000099,1551,"footprint",6.0,true],[940000100,2207,"nearest",3.0,true],[940000101,2207,"uniform",19.0,true],[940000102,2207,"corridor",4.0,true],[940000103,2207,"footprint",6.0,true],[940000104,2209,"nearest",4.0,true],[940000105,2209,"uniform",16.0,true],[940000106,2209,"corridor",4.0,true],[940000107,2209,"footprint",6.0,true],[940000108,2247,"nearest",3.0,true],[940000109,2247,"uniform",14.0,true],[940000110,2247,"corridor",5.0,false],[940000111,2247,"footprint",6.0,true],[940000112,2248,"nearest",5.0,true],[940000113,2248,"uniform",22.0,true],[940000114,2248,"corridor",4.0,false],[940000115,2248,"footprint",6.0,true],[940000116,2280,"nearest",1.0,true],[940000117,2280,"uniform",17.0,true],[940000118,2280,"corridor",5.0,false],[940000119,2280,"footprint",6.0,true],[940000120,2372,"nearest",2.0,true],[940000121,2372,"uniform",12.0,true],[940000122,2372,"corridor",4.0,true],[940000123,2372,"footprint",6.0,true],[940000124,2395,"nearest",3.0,true],[940000125,2395,"uniform",15.0,true],[940000126,2395,"corridor",4.0,true],[940000127,2395,"footprint",6.0,true],[940000128,3225,"nearest",3.0,true],[940000129,3225,"uniform",22.0,true],[940000130,3225,"corridor",4.0,true],[940000131,3225,"footprint",6.0,true],[940000132,3259,"nearest",5.0,true],[940000133,3259,"uniform",21.0,true],[940000134,3259,"corridor",4.0,false],[940000135,3259,"footprint",6.0,true],[940000136,3274,"nearest",4.0,true],[940000137,3274,"uniform",15.0,true],[940000138,3274,"corridor",4.0,true],[940000139,3274,"footprint",5.0,true],[940000140,3301,"nearest",2.0,true],[940000141,3301,"uniform",18.0,true],[940000142,3301,"corridor",4.0,true],[940000143,3301,"footprint",6.0,true],[940000144,3339,"nearest",2.0,false],[940000145,3339,"uniform",17.0,true],[940000146,3339,"corridor",4.0,true],[940000147,3339,"footprint",6.0,true],[940000148,3814,"nearest",3.0,false],[940000149,3814,"uniform",23.0,true],[940000150,3814,"corridor",4.0,true],[940000151,3814,"footprint",6.0,false],[940000152,3881,"nearest",2.0,true],[940000153,3881,"uniform",16.0,true],[940000154,3881,"corridor",4.0,true],[940000155,3881,"footprint",6.0,true],[940000156,3891,"nearest",3.0,true],[940000157,3891,"uniform",17.0,true],[940000158,3891,"corridor",5.0,true],[940000159,3891,"footprint",6.0,false],[940000160,3898,"nearest",5.0,false],[940000161,3898,"uniform",22.0,true],[940000162,3898,"corridor",4.0,false],[940000163,3898,"footprint",6.0,true],[940000164,3908,"nearest",2.0,false],[940000165,3908,"uniform",22.0,true],[940000166,3908,"corridor",4.0,true],[940000167,3908,"footprint",6.0,true],[940000168,3935,"nearest",4.0,false],[940000169,3935,"uniform",20.0,true],[940000170,3935,"corridor",6.0,false],[940000171,3935,"footprint",6.0,true],[940000172,3940,"nearest",3.0,true],[940000173,3940,"uniform",17.0,true],[940000174,3940,"corridor",4.0,true],[940000175,3940,"footprint",6.0,true],[940000176,3957,"nearest",3.0,true],[940000177,3957,"uniform",20.0,true],[940000178,3957,"corridor",4.0,true],[940000179,3957,"footprint",6.0,true],[940000180,3982,"nearest",2.0,true],[940000181,3982,"uniform",17.0,true],[940000182,3982,"corridor",4.0,true],[940000183,3982,"footprint",6.0,true],[940000184,3997,"nearest",5.0,false],[940000185,3997,"uniform",19.0,true],[940000186,3997,"corridor",4.0,false],[940000187,3997,"footprint",6.0,true],[940000188,4658,"nearest",4.0,true],[940000189,4658,"uniform",20.0,false],[940000190,4658,"corridor",4.0,true],[940000191,4658,"footprint",6.0,true],[940000192,4673,"nearest",2.0,true],[940000193,4673,"uniform",15.0,true],[940000194,4673,"corridor",4.0,false],[940000195,4673,"footprint",6.0,false],[940000196,4807,"nearest",2.0,true],[940000197,4807,"uniform",16.0,true],[940000198,4807,"corridor",4.0,true],[940000199,4807,"footprint",6.0,true],[940000200,4820,"nearest",4.0,true],[940000201,4820,"uniform",22.0,true],[940000202,4820,"corridor",4.0,true],[940000203,4820,"footprint",6.0,true],[940000204,4894,"nearest",3.0,true],[940000205,4894,"uniform",25.0,true],[940000206,4894,"corridor",4.0,true],[940000207,4894,"footprint",5.0,true],[940000208,4936,"nearest",3.0,true],[940000209,4936,"uniform",18.0,true],[940000210,4936,"corridor",4.0,true],[940000211,4936,"footprint",5.0,true],[940000212,4975,"nearest",2.0,false],[940000213,4975,"uniform",16.0,true],[940000214,4975,"corridor",4.0,true],[940000215,4975,"footprint",6.0,false],[940000216,4999,"nearest",3.0,false],[940000217,4999,"uniform",18.0,true],[940000218,4999,"corridor",5.0,true],[940000219,4999,"footprint",6.0,true],[940000220,5216,"nearest",3.0,false],[940000221,5216,"uniform",15.0,true],[940000222,5216,"corridor",6.0,true],[940000223,5216,"footprint",6.0,true],[940000224,5279,"nearest",3.0,true],[940000225,5279,"uniform",20.0,true],[940000226,5279,"corridor",4.0,false],[940000227,5279,"footprint",6.0,true],[940000228,5290,"nearest",5.0,false],[940000229,5290,"uniform",17.0,true],[940000230,5290,"corridor",4.0,true],[940000231,5290,"footprint",6.0,true],[940000232,5318,"nearest",5.0,false],[940000233,5318,"uniform",23.0,true],[940000234,5318,"corridor",5.0,false],[940000235,5318,"footprint",6.0,true],[940000236,5332,"nearest",3.0,true],[940000237,5332,"uniform",21.0,true],[940000238,5332,"corridor",4.0,true],[940000239,5332,"footprint",6.0,true],[940000240,5334,"nearest",3.0,true],[940000241,5334,"uniform",19.0,true],[940000242,5334,"corridor",4.0,true],[940000243,5334,"footprint",6.0,true],[940000244,5379,"nearest",2.0,false],[940000245,5379,"uniform",14.0,true],[940000246,5379,"corridor",6.0,true],[940000247,5379,"footprint",6.0,false],[940000248,5397,"nearest",7.0,true],[940000249,5397,"uniform",20.0,true],[940000250,5397,"corridor",4.0,true],[940000251,5397,"footprint",6.0,true],[940000252,5877,"nearest",3.0,false],[940000253,5877,"uniform",19.0,true],[940000254,5877,"corridor",4.0,true],[940000255,5877,"footprint",6.0,true],[940000256,5897,"nearest",2.0,true],[940000257,5897,"uniform",19.0,true],[940000258,5897,"corridor",4.0,true],[940000259,5897,"footprint",6.0,true],[940000260,5902,"nearest",2.0,true],[940000261,5902,"uniform",20.0,true],[940000262,5902,"corridor",4.0,true],[940000263,5902,"footprint",6.0,true],[940000264,5968,"nearest",2.0,true],[940000265,5968,"uniform",17.0,true],[940000266,5968,"corridor",7.0,false],[940000267,5968,"footprint",6.0,true],[940000268,5973,"nearest",2.0,true],[940000269,5973,"uniform",19.0,true],[940000270,5973,"corridor",5.0,false],[940000271,5973,"footprint",6.0,true],[940000272,6029,"nearest",3.0,true],[940000273,6029,"uniform",19.0,true],[940000274,6029,"corridor",4.0,true],[940000275,6029,"footprint",6.0,true],[940000276,6074,"nearest",4.0,false],[940000277,6074,"uniform",16.0,true],[940000278,6074,"corridor",5.0,true],[940000279,6074,"footprint",5.0,true],[940000280,6082,"nearest",3.0,true],[940000281,6082,"uniform",16.0,true],[940000282,6082,"corridor",4.0,true],[940000283,6082,"footprint",6.0,true],[940000284,6089,"nearest",4.0,true],[940000285,6089,"uniform",16.0,true],[940000286,6089,"corridor",5.0,true],[940000287,6089,"footprint",6.0,true],[940000288,6135,"nearest",2.0,true],[940000289,6135,"uniform",21.0,true],[940000290,6135,"corridor",4.0,true],[940000291,6135,"footprint",6.0,true],[940000292,6151,"nearest",4.0,true],[940000293,6151,"uniform",17.0,true],[940000294,6151,"corridor",6.0,false],[940000295,6151,"footprint",6.0,true],[940000296,6171,"nearest",2.0,false],[940000297,6171,"uniform",15.0,true],[940000298,6171,"corridor",4.0,true],[940000299,6171,"footprint",6.0,true],[940000300,6226,"nearest",3.0,true],[940000301,6226,"uniform",19.0,true],[940000302,6226,"corridor",4.0,true],[940000303,6226,"footprint",6.0,true],[940000304,6241,"nearest",4.0,true],[940000305,6241,"uniform",15.0,true],[940000306,6241,"corridor",5.0,true],[940000307,6241,"footprint",6.0,true],[940000308,6273,"nearest",3.0,true],[940000309,6273,"uniform",21.0,true],[940000310,6273,"corridor",4.0,true],[940000311,6273,"footprint",6.0,true],[940000312,6286,"nearest",3.0,true],[940000313,6286,"uniform",20.0,true],[940000314,6286,"corridor",4.0,true],[940000315,6286,"footprint",6.0,true],[940000316,6308,"nearest",4.0,false],[940000317,6308,"uniform",17.0,true],[940000318,6308,"corridor",4.0,true],[940000319,6308,"footprint",6.0,true],[940000320,6324,"nearest",2.0,true],[940000321,6324,"uniform",17.0,true],[940000322,6324,"corridor",4.0,true],[940000323,6324,"footprint",6.0,true],[940000324,6386,"nearest",3.0,true],[940000325,6386,"uniform",20.0,true],[940000326,6386,"corridor",5.0,true],[940000327,6386,"footprint",6.0,false],[940000328,7230,"nearest",3.0,true],[940000329,7230,"uniform",17.0,true],[940000330,7230,"corridor",7.0,false],[940000331,7230,"footprint",6.0,true],[940000332,7248,"nearest",2.0,true],[940000333,7248,"uniform",16.0,true],[940000334,7248,"corridor",4.0,true],[940000335,7248,"footprint",6.0,true],[940000336,7250,"nearest",2.0,true],[940000337,7250,"uniform",13.0,true],[940000338,7250,"corridor",4.0,true],[940000339,7250,"footprint",6.0,true],[940000340,7252,"nearest",2.0,true],[940000341,7252,"uniform",19.0,true],[940000342,7252,"corridor",6.0,false],[940000343,7252,"footprint",6.0,true],[940000344,7257,"nearest",3.0,true],[940000345,7257,"uniform",16.0,true],[940000346,7257,"corridor",4.0,true],[940000347,7257,"footprint",6.0,false],[940000348,7263,"nearest",4.0,false],[940000349,7263,"uniform",20.0,true],[940000350,7263,"corridor",4.0,true],[940000351,7263,"footprint",6.0,true],[940000352,7264,"nearest",5.0,false],[940000353,7264,"uniform",20.0,true],[940000354,7264,"corridor",4.0,false],[940000355,7264,"footprint",6.0,true],[940000356,7398,"nearest",3.0,true],[940000357,7398,"uniform",18.0,true],[940000358,7398,"corridor",4.0,false],[940000359,7398,"footprint",6.0,true],[940000360,7611,"nearest",3.0,false],[940000361,7611,"uniform",17.0,true],[940000362,7611,"corridor",4.0,true],[940000363,7611,"footprint",6.0,true],[940000364,7627,"nearest",3.0,true],[940000365,7627,"uniform",21.0,true],[940000366,7627,"corridor",4.0,true],[940000367,7627,"footprint",5.0,true],[940000368,7676,"nearest",4.0,false],[940000369,7676,"uniform",14.0,true],[940000370,7676,"corridor",5.0,false],[940000371,7676,"footprint",6.0,true],[940000372,7743,"nearest",1.0,true],[940000373,7743,"uniform",22.0,true],[940000374,7743,"corridor",4.0,true],[940000375,7743,"footprint",6.0,true],[940000376,7763,"nearest",2.0,false],[940000377,7763,"uniform",19.0,true],[940000378,7763,"corridor",4.0,false],[940000379,7763,"footprint",6.0,true],[940000380,7770,"nearest",2.0,true],[940000381,7770,"uniform",18.0,true],[940000382,7770,"corridor",4.0,false],[940000383,7770,"footprint",6.0,true],[940000384,7775,"nearest",3.0,true],[940000385,7775,"uniform",15.0,true],[940000386,7775,"corridor",4.0,true],[940000387,7775,"footprint",6.0,true],[940000388,7790,"nearest",2.0,false],[940000389,7790,"uniform",21.0,true],[940000390,7790,"corridor",4.0,false],[940000391,7790,"footprint",6.0,false],[940000392,7908,"nearest",3.0,false],[940000393,7908,"uniform",18.0,true],[940000394,7908,"corridor",5.0,true],[940000395,7908,"footprint",6.0,true],[940000396,7918,"nearest",1.0,true],[940000397,7918,"uniform",16.0,true],[940000398,7918,"corridor",4.0,true],[940000399,7918,"footprint",6.0,true],[940000400,7920,"nearest",3.0,true],[940000401,7920,"uniform",19.0,true],[940000402,7920,"corridor",4.0,true],[940000403,7920,"footprint",6.0,true],[940000404,7923,"nearest",3.0,true],[940000405,7923,"uniform",14.0,true],[940000406,7923,"corridor",4.0,true],[940000407,7923,"footprint",6.0,true],[940000408,7946,"nearest",2.0,true],[940000409,7946,"uniform",17.0,true],[940000410,7946,"corridor",4.0,false],[940000411,7946,"footprint",6.0,true],[940000412,7955,"nearest",2.0,false],[940000413,7955,"uniform",19.0,true],[940000414,7955,"corridor",5.0,true],[940000415,7955,"footprint",6.0,true],[940000416,7985,"nearest",3.0,false],[940000417,7985,"uniform",18.0,true],[940000418,7985,"corridor",4.0,true],[940000419,7985,"footprint",6.0,true],[940000420,8041,"nearest",3.0,true],[940000421,8041,"uniform",17.0,true],[940000422,8041,"corridor",4.0,true],[940000423,8041,"footprint",6.0,true],[940000424,8058,"nearest",4.0,true],[940000425,8058,"uniform",20.0,true],[940000426,8058,"corridor",4.0,true],[940000427,8058,"footprint",6.0,true],[940000428,8099,"nearest",2.0,false],[940000429,8099,"uniform",21.0,true],[940000430,8099,"corridor",4.0,true],[940000431,8099,"footprint",6.0,true],[940000432,8123,"nearest",2.0,true],[940000433,8123,"uniform",20.0,true],[940000434,8123,"corridor",5.0,true],[940000435,8123,"footprint",6.0,true],[940000436,8141,"nearest",2.0,true],[940000437,8141,"uniform",16.0,true],[940000438,8141,"corridor",4.0,true],[940000439,8141,"footprint",6.0,false],[940000440,8155,"nearest",2.0,true],[940000441,8155,"uniform",23.0,true],[940000442,8155,"corridor",4.0,true],[940000443,8155,"footprint",6.0,true],[940000444,8180,"nearest",4.0,true],[940000445,8180,"uniform",20.0,true],[940000446,8180,"corridor",4.0,true],[940000447,8180,"footprint",6.0,true],[940000448,8248,"nearest",4.0,true],[940000449,8248,"uniform",18.0,true],[940000450,8248,"corridor",4.0,false],[940000451,8248,"footprint",6.0,true],[940000452,8299,"nearest",2.0,true],[940000453,8299,"uniform",16.0,true],[940000454,8299,"corridor",4.0,true],[940000455,8299,"footprint",6.0,true],[940000456,8343,"nearest",2.0,true],[940000457,8343,"uniform",14.0,true],[940000458,8343,"corridor",4.0,true],[940000459,8343,"footprint",6.0,true],[940000460,8380,"nearest",3.0,false],[940000461,8380,"uniform",15.0,true],[940000462,8380,"corridor",4.0,true],[940000463,8380,"footprint",6.0,true],[940000464,8460,"nearest",5.0,true],[940000465,8460,"uniform",20.0,true],[940000466,8460,"corridor",4.0,true],[940000467,8460,"footprint",6.0,true],[940000468,8522,"nearest",4.0,true],[940000469,8522,"uniform",20.0,true],[940000470,8522,"corridor",4.0,false],[940000471,8522,"footprint",6.0,true],[940000472,8588,"nearest",3.0,true],[940000473,8588,"uniform",13.0,true],[940000474,8588,"corridor",5.0,false],[940000475,8588,"footprint",6.0,false],[940000476,8594,"nearest",3.0,true],[940000477,8594,"uniform",15.0,true],[940000478,8594,"corridor",4.0,true],[940000479,8594,"footprint",6.0,true]],"recovered_sid":940000368,"hard":{"level":20,"scenario":2749}}'

In [ ]:
# install and import (CPU runtime is enough for this notebook)

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# PYPOWER (the Python port of MATPOWER): the independent implementation used for the validation
import importlib.util, subprocess, sys
if importlib.util.find_spec("pypower") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypower"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("pip install pypower failed: " + r.stderr[-800:])
import pypower
from importlib.metadata import version as _pkg_version
print("PYPOWER", _pkg_version("PYPOWER"))

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""K1 core: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC truth along lambda up to the point where the AC equations stop having a solution (the
"nose"), a signature that the nose is genuine, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the step-1/6 core and power_flow_jacobian from the J1 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # the storm has no solution at lambda = 1: go down until one appears
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Drops toward 0 at a genuine nose."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""N core (reviewer items 4, 5, 9): generator reactive limits inside the AC truth and the check; stress patterns other
than the K1 scaling (loads only, reactive demand only, area-to-area transfer, generator outage on top of the storm);
warm-start timing of the classical solver. Uses LambdaCase / find_lambda_max / truth_below_nose from the K1 core and
build_ybus / newton / klu_solve / system_of / dc_angles / SN_MVA / HAVE_KLU from the step-1/6 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu


# ================================================================ reactive limits
def bus_q_limits(d, gen_rows):
    """Per-bus reactive limits (p.u.) from the generator table of one scenario: sums over in-service generators.
    gen_rows: DataFrame with bus, min_q_mvar, max_q_mvar, in_service, is_slack_gen. Buses without a generator get 0/0."""
    nb = d["nb"]
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    g = gen_rows[gen_rows["in_service"].to_numpy() > 0.5] if "in_service" in gen_rows else gen_rows
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float) / SN_MVA)
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float) / SN_MVA)
    return qmin, qmax


def q_generation(Y, V, Qd_pu):
    """Reactive generation implied by a voltage state (p.u.): injected Q plus the reactive load."""
    return (V * np.conj(Y @ V)).imag + Qd_pu


class QLimCase(LambdaCase):
    """A LambdaCase whose solver enforces generator reactive limits: a PV bus whose generation would exceed its limit
    is held at the limit as a PQ bus (and released again if its voltage recovers past the set-point). The outer loop
    is the standard one used in power-flow programs; a case counts as solved only when no further switching is needed."""

    def __init__(self, d, qmin_pu, qmax_pu, max_outer=15, eps=1e-6):
        super().__init__(d)
        self.qmin, self.qmax, self.max_outer, self.eps = qmin_pu, qmax_pu, max_outer, eps
        self.vset = d["Vm"].copy()                      # generator set-points (the base solution holds them)
        self.limited = self.PV & (self.qmax >= self.qmin)
        self.last = {"switched": {}, "outer": 0, "iters": 0}

    def _inner(self, S, V0, pv, pq, max_iter, tol):
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, pv, pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson with reactive-limit switching. Returns V, converged, total inner iterations, seconds.
        `self.last` keeps the switched buses of the last call ({bus: 'max' | 'min'}) and the number of outer rounds."""
        t0 = time.perf_counter()
        S_base = self.S(lam)
        Qd = lam * self.Qd
        V = np.asarray(V0, complex).copy()
        switched, backs, total_it = {}, {}, 0
        for outer in range(1, self.max_outer + 1):
            sw = np.array(sorted(switched), dtype=int)
            pv = np.array([b for b in self.pv if b not in switched], dtype=int)
            pq = np.array(sorted(set(self.pq.tolist()) | set(switched)), dtype=int)
            S = S_base.copy()
            if len(sw):
                S[sw] = S_base.real[sw] + 1j * (np.array([self.qmax[b] if switched[b] == "max" else self.qmin[b] for b in sw]) - Qd[sw])
            V[pv] = self.vset[pv] * np.exp(1j * np.angle(V[pv]))      # PV buses start at their set-point
            V, ok, it, _ = self._inner(S, V, pv, pq, max_iter, tol)
            total_it += it
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, False, total_it, time.perf_counter() - t0
            Qg = q_generation(self.Y, V, Qd)
            to_max = [b for b in pv if self.limited[b] and Qg[b] > self.qmax[b] + self.eps]
            to_min = [b for b in pv if self.limited[b] and Qg[b] < self.qmin[b] - self.eps]
            Vm = np.abs(V)
            back = [b for b in sw if backs.get(b, 0) < 2 and ((switched[b] == "max" and Vm[b] > self.vset[b] + self.eps) or
                                                             (switched[b] == "min" and Vm[b] < self.vset[b] - self.eps))]
            if not to_max and not to_min and not back:
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, True, total_it, time.perf_counter() - t0
            for b in to_max:
                switched[int(b)] = "max"
            for b in to_min:
                switched[int(b)] = "min"
            for b in back:
                del switched[int(b)]; backs[int(b)] = backs.get(int(b), 0) + 1
        self.last = {"switched": dict(switched), "outer": self.max_outer, "iters": total_it}
        return V, False, total_it, time.perf_counter() - t0

    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        """As LambdaCase.decisions, but voltage limits are checked at every non-slack bus (a generator at its limit no
        longer holds its voltage) and the reactive generation beyond the limits is reported."""
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        chk = ~self.REF
        low = chk & (Vm < self.vmin + margin_v)
        high = chk & (Vm > self.vmax - margin_v)
        vm_chk = np.where(chk, Vm, np.inf)
        Qg = q_generation(self.Y, V, lam * self.Qd)
        over_q = self.limited & (Qg > self.qmax + 1e-6)
        return {"min_vm": float(vm_chk.min()), "weak_bus": int(np.argmin(vm_chk)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA),
                "n_gen_over_qmax": int(over_q.sum()), "q_over_qmax_Mvar": float(SN_MVA * np.clip(Qg - self.qmax, 0, None)[self.limited].sum())}


# ================================================================ stress patterns
class StressCase(LambdaCase):
    """A storm topology stressed in one direction by a factor lambda (1 = the actual operating point):
    'all'      every load and every non-slack generator scaled together (the K1 pattern; slack takes the rest);
    'load'     loads only (P and Q at constant power factor); generation fixed, the slack takes the growth;
    'reactive' reactive demand only (Q of every load); active power unchanged;
    'transfer' loads inside an area scaled, generation outside the area scaled up by the same MW (a transfer into
               the storm area); the slack takes the losses."""

    def __init__(self, d, mode="all", area=None, kappa=None):
        super().__init__(d)
        self.mode = mode
        self.Pg, self.Pd = d["Pg"] / SN_MVA, d["Pd"] / SN_MVA
        self.area = np.zeros(self.nb, bool) if area is None else np.asarray(area, bool)
        self.genB = (self.Pg > 0) & ~self.area & ~self.REF
        self.kappa = kappa if kappa is not None else (self.Pd[self.area].sum() / max(self.Pg[self.genB].sum(), 1e-9))

    def S(self, lam):
        if self.mode == "all":
            return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "load":
            return (self.Pg - lam * self.Pd) + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "reactive":
            return self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "transfer":
            fl = 1.0 + (lam - 1.0) * self.area
            fg = 1.0 + (lam - 1.0) * self.kappa * self.genB
            return (self.Pg * fg - self.Pd * fl) + 1j * (self.Qg - self.Qd * fl)
        raise ValueError(self.mode)

    def dc_loading(self, lam):
        """DC loading at this lambda: angles from the scaled active injections through the DC susceptance matrix."""
        d = self.d
        nb, on = d["nb"], d["on"]
        f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
        B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
        keep = self.nonref
        th = np.zeros(nb)
        th[keep] = splu(B[keep][:, keep]).solve(self.S(lam).real[keep])
        return self.loading(np.exp(1j * (th + np.radians(float(d["Va"][self.ref[0]])))))


def storm_area(d, footprint_buses, min_load_share=0.10, max_hops=6):
    """Buses within h hops of the storm footprint, h grown until the area holds at least `min_load_share` of the load."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=np.asarray(footprint_buses, int), min_only=True)
    total = max(d["Pd"].sum(), 1e-9)
    for h in range(0, max_hops + 1):
        area = hop <= h
        if d["Pd"][area].sum() >= min_load_share * total:
            return area, h, hop
    return hop <= max_hops, max_hops, hop


def footprint_of(d, removed_rows):
    """Endpoint buses of the removed lines (branch-row indices)."""
    return np.unique(np.r_[d["f"][removed_rows], d["t"][removed_rows]])


def pick_generator_outage(d, gen_rows, hop, max_hops=3):
    """The largest in-service non-slack generator (by active output; by reactive range if none produces active power)
    whose bus lies within `max_hops` of the storm footprint; the largest anywhere if there is none. Returns its row
    label in gen_rows (or None)."""
    g = gen_rows
    keep = np.ones(len(g), bool)
    if "in_service" in g:
        keep &= g["in_service"].to_numpy(float) > 0.5
    if "is_slack_gen" in g:
        keep &= g["is_slack_gen"].to_numpy(float) < 0.5
    keep &= ~d["REF"][g["bus"].to_numpy().astype(int)]
    g = g[keep]
    if len(g) == 0:
        return None
    size = g["p_mw"].to_numpy(float)
    if not (size > 0).any():
        size = g["max_q_mvar"].to_numpy(float) - g["min_q_mvar"].to_numpy(float)
    near = hop[g["bus"].to_numpy().astype(int)] <= max_hops
    cand = g[near] if near.any() else g
    size_c = size[near] if near.any() else size
    return int(cand.index[int(np.argmax(size_c))])


def remove_generator(d, gen_rows, row):
    """The truth model after one generator is lost: its P and Q leave the bus (the slack takes the P); if no in-service
    generator remains at the bus, it becomes a PQ bus. Returns the new dict and the reduced generator table."""
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    g = gen_rows.copy()
    bus = int(g.loc[row, "bus"])
    d2["Pg"][bus] -= float(g.loc[row, "p_mw"]); d2["Qg"][bus] -= float(g.loc[row, "q_mvar"])
    g = g.drop(index=row)
    left = g[(g["bus"].to_numpy().astype(int) == bus) & (g["in_service"].to_numpy() > 0.5)] if "in_service" in g else g[g["bus"] == bus]
    if len(left) == 0 and not d2["REF"][bus]:
        d2["PV"][bus] = False; d2["PQ"][bus] = True
        d2["Pg"][bus] = 0.0; d2["Qg"][bus] = 0.0
    return d2, g, bus


# ================================================================ warm starts (item 9)
def timed_solves(case, lam, starts, max_iter=30, tol=1e-8, reps=3):
    """Iterations and the median time (ms) of the solver from each start; converged flag per start."""
    out = {}
    for name, V0 in starts:
        its, ts, oks = [], [], []
        for _ in range(reps):
            V, ok, it, dt = case.solve(lam, V0, max_iter, tol)
            its.append(it); ts.append(1000 * dt); oks.append(bool(ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2))
        out[name] = {"converged": all(oks), "iters": int(np.median(its)), "ms": float(np.median(ts))}
    return out


"""V core: an independent implementation of the AC power-flow equations for validation. PYPOWER (the Python port of
MATPOWER) builds its own admittance matrix from the raw branch parameters of the released tables (r, x, b, tap, shift,
status) and the bus shunts (GS, BS), its own injections from the bus loads and the generator table, and solves with its
own Newton-Raphson (and its own reactive-limit loop). Nothing of our custom Ybus (built from the stored Yff/Yft/Ytf/Ytt
entries), our Python Newton or lightsim2grid is used here, so an agreement between the two is evidence against a shared
modelling error. Uses read_cols / find_local / SN_MVA from the step-1 core."""
import time

import numpy as np
import pandas as pd
from pypower.api import ppoption, runpf
from pypower.bustypes import bustypes
from pypower.dcpf import dcpf
from pypower.makeBdc import makeBdc
from pypower.makeSbus import makeSbus
from pypower.makeYbus import makeYbus
from pypower.newtonpf import newtonpf
from pypower.idx_bus import BUS_I, BUS_TYPE, PD, QD, GS, BS, BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN
from pypower.idx_gen import GEN_BUS, PG, QG, QMAX, QMIN, VG, MBASE, GEN_STATUS, PMAX, PMIN
from pypower.idx_brch import F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A, RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, ANGMIN, ANGMAX

RAW_BUS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "vn_kv", "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc"]
RAW_BR = ["scenario", "idx", "from_bus", "to_bus", "r", "x", "b", "tap", "shift", "rate_a", "br_status", "ang_min", "ang_max",
          "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i", "Ytt_r", "Ytt_i"]
RAW_GEN = ["scenario", "idx", "bus", "p_mw", "q_mvar", "min_p_mw", "max_p_mw", "min_q_mvar", "max_q_mvar", "in_service", "is_slack_gen"]
PP_OPT = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=1)


RAW_CACHE = {}


def preload_raw(local_dir, level, ids):
    """Read the raw tables of many scenarios of one level at once (one pass over the parquet files) into RAW_CACHE."""
    ids = np.unique(np.asarray(ids, dtype=np.int64))
    have = RAW_CACHE.get(level)
    if have is not None:
        ids = ids[~np.isin(ids, have["ids"])]
        if len(ids) == 0:
            return
    bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=ids)
    br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=ids)
    gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=ids)
    for t in (bus, br, gen):
        t["scenario"] = t["scenario"].astype(np.int64)
    if have is None:
        RAW_CACHE[level] = {"ids": ids, "bus": bus, "br": br, "gen": gen}
    else:
        RAW_CACHE[level] = {"ids": np.r_[have["ids"], ids], "bus": pd.concat([have["bus"], bus]), "br": pd.concat([have["br"], br]), "gen": pd.concat([have["gen"], gen])}
    c = RAW_CACHE[level]
    c["g_bus"], c["g_br"], c["g_gen"] = c["bus"].groupby("scenario"), c["br"].groupby("scenario"), c["gen"].groupby("scenario")


def raw_tables(local_dir, level, sid):
    """The released bus, branch and generator tables of one scenario, with the raw network parameters."""
    sid = int(sid)
    c = RAW_CACHE.get(level)
    if c is not None and sid in c["g_bus"].groups:
        bus, br, gen = (c[g].get_group(sid).copy() for g in ["g_bus", "g_br", "g_gen"])
    else:
        bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=[sid])
        br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=[sid])
        gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=[sid])
    bus = bus.sort_values("bus").reset_index(drop=True); br = br.sort_values("idx").reset_index(drop=True); gen = gen.sort_values("idx").reset_index(drop=True)
    if len(bus) == 0 or len(br) == 0 or len(gen) == 0:
        raise RuntimeError(f"scenario {sid} of level {level} is not on disk")
    return bus, br, gen


def admittance_check(br):
    """Recompute the branch admittance entries from r, x, b, tap and shift with the MATPOWER formulas and compare with the
    stored Yff/Yft/Ytf/Ytt: a check on the units, the tap and phase-shift conventions and the branch status."""
    on = br["br_status"].to_numpy(float) > 0.5
    ys = on / (br["r"].to_numpy(float) + 1j * br["x"].to_numpy(float))
    bc = on * br["b"].to_numpy(float)
    tap = br["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
    t = tap * np.exp(1j * np.radians(br["shift"].to_numpy(float)))
    Ytt = ys + 1j * bc / 2
    Yff = Ytt / (t * np.conj(t))
    Yft = -ys / np.conj(t)
    Ytf = -ys / t
    stored = {k: br[k + "_r"].to_numpy(float) + 1j * br[k + "_i"].to_numpy(float) for k in ["Yff", "Yft", "Ytf", "Ytt"]}
    diff = {k: float(np.max(np.abs(v - stored[k]))) for k, v in [("Yff", Yff), ("Yft", Yft), ("Ytf", Ytf), ("Ytt", Ytt)]}
    scale = float(np.max(np.abs(stored["Yff"])))
    return {"max_abs_diff": max(diff.values()), "max_rel_diff": max(diff.values()) / scale, "per_entry": diff,
            "branches": int(len(br)), "in_service": int(on.sum()), "transformers_tap_not_1": int((tap != 1.0).sum()),
            "phase_shifters": int((br["shift"].to_numpy(float) != 0).sum()), "zero_impedance": int((np.abs(br["r"] + 1j * br["x"]) == 0).sum())}


class Indep:
    """One scenario's tables as a PYPOWER case. Lines are removed by their idx; the load factor lam scales every load and
    every non-slack generator, as our LambdaCase does (the slack takes the rest); generator voltage set-points are the
    released Vm of their buses and are never changed."""

    def __init__(self, bus, br, gen):
        self.bus, self.br, self.gen = bus.reset_index(drop=True), br.reset_index(drop=True), gen.reset_index(drop=True)
        self.nb = len(self.bus)
        self.PQ, self.PV, self.REF = (self.bus[c].to_numpy(float) > 0.5 for c in ["PQ", "PV", "REF"])
        self.vmin, self.vmax = self.bus["min_vm_pu"].to_numpy(float), self.bus["max_vm_pu"].to_numpy(float)
        self.idx = self.br["idx"].to_numpy().astype(int)
        self.rate = self.br["rate_a"].to_numpy(float)

    # ---------------------------------------------------------------- the case
    def ppc(self, removed_idx=(), lam=1.0, Vm0=None, Va0=None):
        b, g, r = self.bus, self.gen, self.br
        nb, ng, nl = len(b), len(g), len(r)
        bus = np.zeros((nb, 13))
        bus[:, BUS_I] = np.arange(nb)                                   # internal numbering: our buses are already 0..nb-1
        bus[:, BUS_TYPE] = np.where(self.REF, 3, np.where(self.PV, 2, 1))
        bus[:, PD] = lam * b["Pd"].to_numpy(float); bus[:, QD] = lam * b["Qd"].to_numpy(float)
        bus[:, GS] = b["GS"].to_numpy(float) * SN_MVA; bus[:, BS] = b["BS"].to_numpy(float) * SN_MVA   # stored in p.u. (datakit divides by baseMVA)
        bus[:, BUS_AREA] = 1; bus[:, ZONE] = 1
        bus[:, VM] = b["Vm"].to_numpy(float) if Vm0 is None else np.asarray(Vm0, float)
        bus[:, VA] = b["Va"].to_numpy(float) if Va0 is None else np.asarray(Va0, float)
        bus[:, BASE_KV] = b["vn_kv"].to_numpy(float); bus[:, VMAX] = self.vmax; bus[:, VMIN] = self.vmin
        # PV/REF buses of the released tables without an in-service generator (datakit turns such buses into PQ, so the released
        # Texas data should have none; counted in the conventions check): a dummy generator holds their voltage as the flags say,
        # so that PYPOWER models the same bus types as our solver, which trusts the flags
        on = g["in_service"].to_numpy(float) > 0.5
        has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)[on]] = True
        dummy = np.nonzero((self.PV | self.REF) & ~has)[0]
        ng2 = ng + len(dummy)
        gen = np.zeros((ng2, 21))
        gb = np.r_[g["bus"].to_numpy().astype(int), dummy]
        gen[:, GEN_BUS] = gb
        slack = np.r_[g["is_slack_gen"].to_numpy(float) > 0.5, self.REF[dummy]]
        gen[:, PG] = np.r_[g["p_mw"].to_numpy(float), np.zeros(len(dummy))] * np.where(slack, 1.0, lam)
        gen[:, QG] = np.r_[g["q_mvar"].to_numpy(float), b["Qg"].to_numpy(float)[dummy]]
        gen[:, QMAX] = np.r_[g["max_q_mvar"].to_numpy(float), np.full(len(dummy), 1e9)]
        gen[:, QMIN] = np.r_[g["min_q_mvar"].to_numpy(float), np.full(len(dummy), -1e9)]
        gen[:, VG] = b["Vm"].to_numpy(float)[gb]                        # the set-point: the released voltage of the bus
        gen[:, MBASE] = SN_MVA
        gen[:, GEN_STATUS] = np.r_[on, np.ones(len(dummy), bool)].astype(float)
        gen[:, PMAX] = np.r_[g["max_p_mw"].to_numpy(float), np.zeros(len(dummy))]; gen[:, PMIN] = np.r_[g["min_p_mw"].to_numpy(float), np.zeros(len(dummy))]
        self.n_dummy = len(dummy)
        branch = np.zeros((nl, 17))                                     # 13 inputs + 4 flow columns that pfsoln fills
        branch[:, F_BUS] = r["from_bus"].to_numpy().astype(int); branch[:, T_BUS] = r["to_bus"].to_numpy().astype(int)
        branch[:, BR_R] = r["r"].to_numpy(float); branch[:, BR_X] = r["x"].to_numpy(float); branch[:, BR_B] = r["b"].to_numpy(float)
        branch[:, RATE_A] = self.rate
        tap = r["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
        branch[:, TAP] = tap; branch[:, SHIFT] = r["shift"].to_numpy(float)
        status = r["br_status"].to_numpy(float) > 0.5
        if len(removed_idx):
            status = status & ~np.isin(self.idx, np.asarray(removed_idx, dtype=int))
        branch[:, BR_STATUS] = status.astype(float)
        branch[:, ANGMIN] = r["ang_min"].to_numpy(float); branch[:, ANGMAX] = r["ang_max"].to_numpy(float)
        return {"version": "2", "baseMVA": SN_MVA, "bus": bus, "gen": gen, "branch": branch}

    # ---------------------------------------------------------------- starts
    def start_state(self, ppc, start):
        """Initial complex voltage for a start: 'base' (the released state), 'flat', 'dc' (PYPOWER's DC power flow), or a
        (Vm, Va_deg) pair. Generator buses always begin at their set-point magnitude, as in PYPOWER's runpf."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        if isinstance(start, str):
            if start == "base":
                Vm, Va = self.bus["Vm"].to_numpy(float), np.radians(self.bus["Va"].to_numpy(float))
            elif start == "flat":
                Vm, Va = np.ones(self.nb), np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0))
            elif start == "dc":
                Bbus, Bf, Pbusinj, Pfinj = makeBdc(ppc["baseMVA"], bus, ppc["branch"])
                Pbus = makeSbus(ppc["baseMVA"], bus, gen).real - Pbusinj - bus[:, GS] / ppc["baseMVA"]
                Va = dcpf(Bbus, Pbus, np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0)), ref, pv, pq)
                Vm = np.ones(self.nb)
            else:
                raise ValueError(start)
        else:
            Vm, Va = np.asarray(start[0], float), np.radians(np.asarray(start[1], float))
        V0 = Vm * np.exp(1j * Va)
        on = np.nonzero(gen[:, GEN_STATUS] > 0)[0]
        gbus = gen[on, GEN_BUS].astype(int)
        vcb = np.ones(self.nb); vcb[pq] = 0
        k = np.nonzero(vcb[gbus])[0]
        V0[gbus[k]] = gen[on[k], VG] / np.abs(V0[gbus[k]]) * V0[gbus[k]]
        return V0, ref, pv, pq

    # ---------------------------------------------------------------- solves
    def mismatch(self, ppc, V):
        """PYPOWER's power-flow mismatch (largest entry, p.u.) of a voltage state: P at every non-slack bus, Q at PQ buses."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        Ybus, _, _ = makeYbus(ppc["baseMVA"], bus, ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        mis = V * np.conj(Ybus @ V) - Sbus
        F = np.r_[mis[pv].real, mis[pq].real, mis[pq].imag]
        return float(np.max(np.abs(F))) if len(F) else 0.0

    def solve(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8):
        """PYPOWER's Newton-Raphson on the case. Returns a dict: converged, iterations, seconds, mismatch at the start and at
        the end (p.u.), the state (Vm, Va_deg) and the screening decisions made on it."""
        ppc = self.ppc(removed_idx, lam)
        V0, ref, pv, pq = self.start_state(ppc, start)
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], ppc["bus"], ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], ppc["bus"], ppc["gen"])
        m0 = self.mismatch(ppc, V0)
        t0 = time.perf_counter()
        note = ""
        try:
            V, ok, it = newtonpf(Ybus, Sbus, V0, ref, pv, pq, opt)
        except Exception as exc:                      # a singular Jacobian on a diverging path makes SuperLU raise: that is a failed start
            V, ok, it = np.full(len(V0), np.nan, dtype=complex), False, int(max_it)
            note = f"solver error: {type(exc).__name__}"
        sec = time.perf_counter() - t0
        ok = bool(ok) and bool(np.all(np.isfinite(V)))
        out = {"converged": ok, "iterations": int(it), "seconds": sec, "mismatch_start_pu": m0, "note": note,
               "mismatch_end_pu": self.mismatch(ppc, V) if np.all(np.isfinite(V)) else np.inf}
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        return out

    def decisions(self, ppc, Yf, Yt, V, Ybus, Sbus, lam):
        if not np.all(np.isfinite(V)):
            nb = len(V)
            return {"Vm": np.full(nb, np.nan), "Va_deg": np.full(nb, np.nan), "loading": np.full(len(ppc["branch"]), np.nan), "overloaded_idx": [],
                    "n_over": 0, "max_loading": np.nan, "min_vm": np.nan, "weak_bus": -1, "n_low_v": 0, "n_high_v": 0, "low_v_buses": [], "Qg_bus_Mvar": np.full(nb, np.nan)}
        branch = ppc["branch"]
        f, t = branch[:, F_BUS].astype(int), branch[:, T_BUS].astype(int)
        Sf = V[f] * np.conj(Yf @ V) * ppc["baseMVA"]
        St = V[t] * np.conj(Yt @ V) * ppc["baseMVA"]
        rated = (branch[:, BR_STATUS] > 0) & (branch[:, RATE_A] > 0)
        ld = np.full(len(branch), np.nan); ld[rated] = np.maximum(np.abs(Sf), np.abs(St))[rated] / branch[rated, RATE_A]
        Vm = np.abs(V)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        inj = V * np.conj(Ybus @ V)
        Qg_bus = (inj.imag + lam * self.bus["Qd"].to_numpy(float) / ppc["baseMVA"]) * ppc["baseMVA"]
        return {"Vm": Vm, "Va_deg": np.degrees(np.angle(V)), "loading": ld, "overloaded_idx": self.idx[rated & (ld >= 1.0)].tolist(),
                "n_over": int(np.nansum(ld >= 1.0)), "max_loading": float(np.nanmax(ld)) if rated.any() else np.nan,
                "min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int((self.PQ & (Vm < self.vmin)).sum()), "n_high_v": int((self.PQ & (Vm > self.vmax)).sum()),
                "low_v_buses": np.nonzero(self.PQ & ((Vm < self.vmin) | (Vm > self.vmax)))[0].tolist(),
                "Qg_bus_Mvar": Qg_bus}

    def solve_qlim(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8, slack_unlimited=True, convention="matpower",
                   max_rounds=15, eps=1e-6):
        """Reactive-limit enforcement around PYPOWER's Newton-Raphson (PYPOWER's own runpf loop fails on current numpy, so the
        outer loop is written here with PYPOWER's primitives). Two conventions:
          'matpower'  as MATPOWER/PYPOWER: a generator beyond a limit is fixed at that limit and its bus becomes PQ (the other
                      generators at the bus keep the reactive output of the last solution); nothing is converted back;
          'release'   as our notebook N1: limits summed per bus; a PV bus whose total exceeds a limit is held at the sum as a
                      PQ bus, and released again when its voltage recovers past the set-point (at most twice per bus).
        slack_unlimited leaves the slack generator unlimited (as in N1); with False the slack bus is limited like the others and
        a PV bus takes over as slack when it is converted. Returns converged, rounds, the state, the decisions, the buses held at
        a limit and the generators' reactive outputs."""
        from pypower.pfsoln import pfsoln
        ppc = self.ppc(removed_idx, lam)
        bus, gen, branch = ppc["bus"].copy(), ppc["gen"].copy(), ppc["branch"].copy()
        V, ref, pv, pq = self.start_state(ppc, start)
        if slack_unlimited:
            s_ = np.r_[self.gen["is_slack_gen"].to_numpy(float) > 0.5, np.zeros(len(gen) - len(self.gen), bool)]
            gen[s_, QMAX], gen[s_, QMIN] = 1e9, -1e9
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], bus, branch)
        on = gen[:, GEN_STATUS] > 0
        gbus = gen[:, GEN_BUS].astype(int)
        qmin_bus, qmax_bus = np.zeros(self.nb), np.zeros(self.nb)
        np.add.at(qmin_bus, gbus[on], gen[on, QMIN]); np.add.at(qmax_bus, gbus[on], gen[on, QMAX])
        held = {}                       # bus -> 'max' | 'min'
        backs = {}
        t0 = time.perf_counter(); total_it = 0; ok = False; rounds = 0
        vset = gen[:, VG].copy()
        bus_types0 = bus[:, BUS_TYPE].copy()
        for rounds in range(1, max_rounds + 1):
            bus[:, BUS_TYPE] = bus_types0
            for b_ in held:
                bus[b_, BUS_TYPE] = 1
            ref, pv, pq = bustypes(bus, gen)
            if len(ref) == 0:
                break
            # fixed injections of the held buses: generation at the summed limit (the gens' QG entries sum to it)
            for b_, side in held.items():
                gi = np.nonzero(on & (gbus == b_))[0]
                lim = gen[gi, QMAX] if side == "max" else gen[gi, QMIN]
                gen[gi, QG] = lim
            Sbus = makeSbus(ppc["baseMVA"], bus, gen)
            # PV buses start at their set-point magnitude
            for b_ in pv:
                gi = np.nonzero(on & (gbus == b_))[0]
                V[b_] = vset[gi[0]] * np.exp(1j * np.angle(V[b_]))
            try:
                V, ok, it = newtonpf(Ybus, Sbus, V, ref, pv, pq, opt)
            except Exception:
                V, ok, it = np.full(len(V), np.nan, dtype=complex), False, int(max_it)
            total_it += int(it)
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                ok = False
                break
            bus_, gen_, branch_ = pfsoln(ppc["baseMVA"], bus.copy(), gen.copy(), branch.copy(), Ybus, Yf, Yt, V, ref, pv, pq)
            Qg_bus = np.zeros(self.nb); np.add.at(Qg_bus, gbus[on], gen_[on, QG])
            Vm = np.abs(V)
            if convention == "matpower":
                viol_max = on & (gen_[:, QG] > gen_[:, QMAX] + eps) & np.isin(gbus, pv)
                viol_min = on & (gen_[:, QG] < gen_[:, QMIN] - eps) & np.isin(gbus, pv)
                to_max = sorted(set(gbus[viol_max].tolist())); to_min = sorted(set(gbus[viol_min].tolist()) - set(to_max))
                back = []
            else:
                to_max = [b_ for b_ in pv if Qg_bus[b_] > qmax_bus[b_] + eps and qmax_bus[b_] >= qmin_bus[b_]]
                to_min = [b_ for b_ in pv if Qg_bus[b_] < qmin_bus[b_] - eps and qmax_bus[b_] >= qmin_bus[b_]]
                back = [b_ for b_, side in held.items() if backs.get(b_, 0) < 2 and
                        ((side == "max" and Vm[b_] > vset[np.nonzero(gbus == b_)[0][0]] + eps) or
                         (side == "min" and Vm[b_] < vset[np.nonzero(gbus == b_)[0][0]] - eps))]
            if not to_max and not to_min and not back:
                gen = gen_
                break
            ok = False
            for b_ in to_max:
                held[int(b_)] = "max"
            for b_ in to_min:
                held[int(b_)] = "min"
            for b_ in back:
                del held[int(b_)]; backs[int(b_)] = backs.get(int(b_), 0) + 1
            if convention == "matpower":
                # the generators of a converted bus that are not themselves beyond a limit keep their last reactive output
                for b_ in list(to_max) + list(to_min):
                    gi = np.nonzero(on & (gbus == b_))[0]
                    for g_ in gi:
                        if not (viol_max[g_] or viol_min[g_]):
                            gen[g_, QG] = gen_[g_, QG]
                            gen[g_, QMAX] = gen[g_, QMIN] = gen_[g_, QG]      # frozen at that value for the rest of the loop
        sec = time.perf_counter() - t0
        out = {"converged": bool(ok), "rounds": int(rounds), "inner_iterations": int(total_it), "seconds": sec}
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        out.update({"buses_at_limit": sorted(held), "held": dict(held), "Qg_gen_Mvar": gen[:, QG].copy(), "convention": convention})
        return out


def complementarity(bus_df, gen_df, Vm, Qg_bus_Mvar, eps_q=1e-3, eps_v=1e-6):
    """Reactive-limit complementarity of a state, per PV bus with limits (per-bus aggregate of its in-service generators):
    within limits -> the voltage must sit at its set-point; at Qmax -> voltage at or below the set-point; at Qmin -> at or
    above. Returns counts of buses in each condition and of violations."""
    g = gen_df[gen_df["in_service"].to_numpy(float) > 0.5]
    nb = len(bus_df)
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float))
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float))
    has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)] = True
    pv = (bus_df["PV"].to_numpy(float) > 0.5) & has & (qmax >= qmin)
    vset = bus_df["Vm"].to_numpy(float)
    q = np.asarray(Qg_bus_Mvar, float); v = np.asarray(Vm, float)
    inside = pv & (q < qmax - eps_q) & (q > qmin + eps_q)
    at_max = pv & (q >= qmax - eps_q); at_min = pv & (q <= qmin + eps_q)
    beyond = pv & ((q > qmax + eps_q) | (q < qmin - eps_q))
    bad = (inside & (np.abs(v - vset) > eps_v)) | (at_max & (v > vset + eps_v)) | (at_min & (v < vset - eps_v)) | beyond
    return {"pv_buses_with_limits": int(pv.sum()), "inside": int(inside.sum()), "at_qmax": int(at_max.sum()), "at_qmin": int(at_min.sum()),
            "beyond_limits": int(beyond.sum()), "complementarity_violations": int(bad.sum()), "violating_buses": np.nonzero(bad)[0].tolist()[:20]}


def same_state(Vm_a, Vm_b, tol=1e-4):
    return bool(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b))) < tol)


def state_diff(Vm_a, Va_a, Vm_b, Va_b):
    dva = (np.asarray(Va_a) - np.asarray(Va_b) + 180) % 360 - 180
    return {"max_dVm_pu": float(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b)))), "max_dVa_deg": float(np.max(np.abs(dva)))}


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)


PLAN = json.loads(PLAN_JSON)
print('plan:', {k: (len(v) if isinstance(v, list) else v) for k, v in PLAN.items()})

## 1. Download: model, the two N-2 data blocks of the base points, and the block of the hard case

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
HARD = {int(PLAN['hard']['level']): [int(PLAN['hard']['scenario'])]}
want = {k: {s // PART for s in v} for k, v in HARD.items()}
want.setdefault(LEVEL_BASE, set()).update(BASE_PARTS)
for k, parts in want.items():
    for p, s in all_files:
        m = RAW3_PAT.search(p) if p.startswith(f"data/test_{k}/") else None
        if m and int(m.group(2)) in parts:
            plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR)
print("Download finished")

## 2. Base points, GENCO on the CPU, and a check of the model

In [ ]:
def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}


base_disk = scenario_ids(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"))
base_pool = np.array([s for s in base_disk if int(s) not in HARD2], dtype=np.int64)
BASE_IDS = np.sort(RNG.choice(base_pool, min(N_BASE, len(base_pool)), replace=False)).astype(np.int64)
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
tot = np.array([d["Pd"].sum() for d in BASE.values()])
print(f"{len(BASE)} N-{LEVEL_BASE} operating points as storm base points | {NB} buses | "
      f"total load {tot.min() / 1000:.1f}-{tot.max() / 1000:.1f} GW")

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


BASE_GRAPHS = [build_graph(int(s), int(s)) for s in BASE_IDS]
print(f"{len(BASE_GRAPHS)} base graphs built")

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
# [GPU] run GENCO on a list of graphs; check the model against the stored AC solutions of the base points

def run_genco(graphs, bs=BATCH):
    """GENCO's (Vm, Va in radians) for each graph, in order."""
    out = [None] * len(graphs)
    for i, h in enumerate(graphs):
        h["scenario_id"] = torch.tensor([i], dtype=torch.long)
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            o = forward(MODEL, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(i)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy().astype(np.float64),
                           o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy().astype(np.float64))
    return out


t0 = time.perf_counter()
BASE_OUT = run_genco(BASE_GRAPHS[:N_CHECK])
T_FWD_MS = 1000 * (time.perf_counter() - t0) / len(BASE_OUT)
errs = np.array([np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va) in zip(BASE_IDS[:N_CHECK], BASE_OUT)])
print(f"GENCO on {len(BASE_OUT)} base points: worst-bus voltage error median {np.median(errs):.4f} p.u., "
      f"95th percentile {np.percentile(errs, 95):.4f} p.u. (expected about 0.001 and 0.004) | {T_FWD_MS:.0f} ms per graph incl. loading")
if np.median(errs) > 0.02 and NB > 100:
    raise RuntimeError("GENCO's error on ordinary N-2 cases is far above the known level: the model did not load correctly. Send me this output.")

In [ ]:
# [GPU] GENCO on a storm at any stress level, and on a storm with a generator removed

def storm_graph(c, bus_df=None, gen_df=None):
    """GENCO's input graph of a storm (optionally with modified bus / generator tables)."""
    b = (BG.get_group(c["base"]) if bus_df is None else bus_df).sort_values("bus").reset_index(drop=True).copy()
    g = GG.get_group(c["base"]) if gen_df is None else gen_df
    r = RG.get_group(c["base"]).copy()
    r.loc[r["idx"].astype(int).isin([int(x) for x in c["removed_lines"]]), "br_status"] = 0.0
    h = build_hetero_data_for_scenario(int(c["sid"]), b, g, r)
    NORMALIZER.transform(h)
    return TRANSFORM(h)


def scaled_graph_mode(g, lam, mode, area=None, kappa=1.0, genB=None):
    """The storm graph with its injections scaled the way StressCase scales them (loads and generation enter the
    model divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    if mode == "all":
        h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    elif mode == "load":
        h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam
    elif mode == "reactive":
        h["bus"].x[:, QD_H] *= lam
    elif mode == "transfer":
        fl = torch.tensor(1.0 + (lam - 1.0) * np.asarray(area, float), dtype=h["bus"].x.dtype)
        h["bus"].x[:, PD_H] *= fl; h["bus"].x[:, QD_H] *= fl
        gen_bus = h["gen", "connected_to", "bus"].edge_index[1].cpu().numpy()
        fg = torch.tensor(1.0 + (lam - 1.0) * kappa * np.asarray(genB, float)[gen_bus], dtype=h["gen"].x.dtype)
        h["gen"].x[:, PG_H] *= fg
    else:
        raise ValueError(mode)
    return h


def genco_state(case, vm, va):
    return case.state(vm, np.degrees(va))


print("GENCO helpers ready")

## 3. Conventions and provenance

The released reference solutions were produced by gridfm-datakit (branch `genco-paper-repro`) with **PowerModels.jl**
(`solve_ac_pf`, Ipopt, tol 10⁻⁶), not with pandapower as the earlier drafts said. The cell below recomputes the stored
branch admittances from r, x, b, tap and shift with the MATPOWER formulas, checks bus types against the generator table,
counts transformers, phase shifters, shunts and buses with several limited generators, and evaluates the released
solutions under PYPOWER's independently built equations (also with the shunts read in the wrong unit, to show that the
unit matters and that the right one was used).

In [ ]:
# Where the released reference solutions come from, and whether the raw parameters, our reading of them and PYPOWER's agree

PROVENANCE = {
    "released_solutions": "gridfm-datakit, branch genco-paper-repro (scripts/data_gen/launcher/contingency/launch_finetuning_data_gen_texas.py "
                          "with pf_fast=False): AC power flow solved by PowerModels.jl solve_ac_pf with Ipopt (tol 1e-6, max_iter 200) on "
                          "Texas2k_case1_2016summerpeak_corrected.m; the PF formulation holds PV voltages and active generation fixed and "
                          "does not enforce generator reactive limits. pandapower is not used on that branch (it was, before the "
                          "'powermodels integration' commit of 29 October 2025).",
    "bus_table_units": "Pd, Qd, Pg, Qg in MW/Mvar; Vm p.u.; Va degrees in [-180, 180); GS, BS divided by baseMVA (p.u.); vn_kv kV; "
                       "PQ/PV/REF one-hot (process_network.py of that branch).",
    "branch_table": "r, x, b in p.u. on baseMVA = 100; tap (1 where the file had 0), shift in degrees; rate_a MVA; br_status; the stored "
                    "Yff/Yft/Ytf/Ytt entries are computed by datakit's branch_vectors (MATPOWER formulas).",
    "our_solvers": "our Python Newton-Raphson and lightsim2grid both use an admittance matrix built from the stored Yff/Yft/Ytf/Ytt and "
                   "GS/BS, injections from the bus table; PYPOWER here builds everything from r, x, b, tap, shift, status, GS, BS and the "
                   "generator table.",
    "pypower_version": _pkg_version("PYPOWER"),
}
t0 = time.perf_counter()
preload_raw(LOCAL_DIR, LEVEL_BASE, BASE_IDS)
print(f"raw tables of {len(BASE_IDS)} base scenarios read in {time.perf_counter() - t0:.0f} s")
rows, mis_rows = [], []
for j, sid in enumerate(BASE_IDS):
    bus, br, gen = raw_tables(LOCAL_DIR, LEVEL_BASE, int(sid))
    chk = admittance_check(br)
    on_g = gen[gen.in_service > 0.5]
    gb = on_g.bus.to_numpy().astype(int)
    has_gen = np.zeros(len(bus), bool); has_gen[gb] = True
    n_per_bus = np.bincount(gb, minlength=len(bus))
    limited = (on_g.max_q_mvar.to_numpy() >= on_g.min_q_mvar.to_numpy())
    multi = np.bincount(gb[limited], minlength=len(bus)) > 1
    PVb, REFb, PQb = bus.PV > 0.5, bus.REF > 0.5, bus.PQ > 0.5
    ind = Indep(bus, br, gen)
    ppc = ind.ppc()
    V_rel = bus.Vm.to_numpy() * np.exp(1j * np.radians(bus.Va.to_numpy()))
    m_pu = ind.mismatch(ppc, V_rel)
    ppc_mw = ind.ppc(); ppc_mw["bus"][:, 4] = bus.GS.to_numpy(); ppc_mw["bus"][:, 5] = bus.BS.to_numpy()     # GS/BS read as MW instead of p.u.
    m_mw = ind.mismatch(ppc_mw, V_rel)
    Pg_sum = np.bincount(gb, weights=on_g.p_mw.to_numpy(float), minlength=len(bus)); Qg_sum = np.bincount(gb, weights=on_g.q_mvar.to_numpy(float), minlength=len(bus))
    rows.append({"sid": int(sid), "max_rel_diff_Y": chk["max_rel_diff"], "transformers": chk["transformers_tap_not_1"], "phase_shifters": chk["phase_shifters"],
                 "gen_sum_vs_bus_Pg_max_diff_MW": float(np.abs(Pg_sum - bus.Pg.to_numpy()).max()), "gen_sum_vs_bus_Qg_max_diff_Mvar": float(np.abs(Qg_sum - bus.Qg.to_numpy()).max()),
                 "zero_impedance_branches": chk["zero_impedance"], "branches_out": int(chk["branches"] - chk["in_service"]),
                 "pv_or_ref_without_gen": int(((PVb | REFb) & ~has_gen).sum()), "pq_with_in_service_gen": int((PQb & has_gen).sum()),
                 "pq_with_pg": int((PQb & (bus.Pg.abs() > 1e-6)).sum()), "slack_gens": int((on_g.is_slack_gen > 0.5).sum()),
                 "ref_buses": int(REFb.sum()), "pv_buses": int(PVb.sum()), "pv_buses_with_several_limited_gens": int((multi & (PVb | REFb)).sum()),
                 "buses_with_shunt": int(((bus.GS != 0) | (bus.BS != 0)).sum()),
                 "released_mismatch_pypower_pu": m_pu, "released_mismatch_if_GS_BS_were_MW_pu": m_mw})
    if j < 20:
        r = ind.solve(start="base", max_it=10)
        mis_rows.append({"sid": int(sid), "iterations_from_released": r["iterations"], "max_dVm_vs_released": float(np.abs(r["Vm"] - bus.Vm.to_numpy()).max()),
                         "max_dVa_vs_released": state_diff(r["Vm"], r["Va_deg"], bus.Vm.to_numpy(), bus.Va.to_numpy())["max_dVa_deg"], "converged": r["converged"]})
P1 = pd.DataFrame(rows); M1 = pd.DataFrame(mis_rows)
P1.to_csv(Path(OUT_DIR) / "conventions.csv", index=False)
print(f"{len(P1)} base scenarios checked in {time.perf_counter() - t0:.0f} s")
print(f"Stored admittance entries against r, x, b, tap, shift recomputed with the MATPOWER formulas: largest relative difference "
      f"{P1.max_rel_diff_Y.max():.1e} (0 means identical).")
print(f"Per scenario: {P1.transformers.median():.0f} branches with tap != 1, {P1.phase_shifters.median():.0f} phase shifters, "
      f"{P1.buses_with_shunt.median():.0f} buses with a shunt, {P1.ref_buses.median():.0f} slack bus / {P1.slack_gens.median():.0f} slack generator, "
      f"{P1.pv_buses.median():.0f} PV buses of which {P1.pv_buses_with_several_limited_gens.median():.0f} hold several limited generators.")
print(f"Generator table against bus table: in-service generators sum to the bus Pg / Qg to within {P1.gen_sum_vs_bus_Pg_max_diff_MW.max():.1e} MW / {P1.gen_sum_vs_bus_Qg_max_diff_Mvar.max():.1e} Mvar.")
print(f"Bus-type consistency: PV/REF buses without an in-service generator {P1.pv_or_ref_without_gen.max()} (max over scenarios); "
      f"PQ buses with an in-service generator {P1.pq_with_in_service_gen.max()}, with Pg != 0: {P1.pq_with_pg.max()}.")
print(f"Released solutions under PYPOWER's equations (GS/BS read as p.u., as datakit writes them): largest mismatch {P1.released_mismatch_pypower_pu.max():.1e} p.u. "
      f"= {100 * P1.released_mismatch_pypower_pu.max():.1e} MW; read as MW instead: {P1.released_mismatch_if_GS_BS_were_MW_pu.max():.1e} p.u.")
if len(M1):
    print(f"PYPOWER started from the released state: converged on {int(M1.converged.sum())} of {len(M1)}, {M1.iterations_from_released.max()} iterations at most, "
          f"and moved the voltages by at most {M1.max_dVm_vs_released.max():.1e} p.u. / {M1.max_dVa_vs_released.max():.1e} degrees: the released states are "
          f"solutions of the independently built equations to that precision.")
display(P1.describe().T[["min", "50%", "max"]])

## 4. Ordinary and near-limit released cases

In [ ]:
# Ordinary and near-limit released N-2 cases: PYPOWER from flat and from the released state, against the released solution and our solver

rng = np.random.default_rng(SEED + 7)
ordinary = np.sort(rng.choice(base_disk, min(N_ORDINARY, len(base_disk)), replace=False)).astype(np.int64)
t0 = time.perf_counter()
EV_ALL = truth_dicts(LEVEL_BASE, base_disk)
near_rows = []
for sid, d in EV_ALL.items():
    ev = ScenarioEval(d)
    vm = d["Vm"][d["PQ"]]; lo = d["min_vm_pu"][d["PQ"]]; hi = d["max_vm_pu"][d["PQ"]]
    near_rows.append({"sid": int(sid), "line_gap": float(np.min(np.abs(ev.ld_t - 1.0))) if len(ev.ld_t) else np.inf,
                      "v_gap": float(min(np.min(np.abs(vm - lo)), np.min(np.abs(vm - hi)))) if len(vm) else np.inf})
NR_ = pd.DataFrame(near_rows)
near = np.array(sorted(set(NR_.nsmallest(N_NEAR // 2, "line_gap").sid.tolist()) | set(NR_.nsmallest(N_NEAR - N_NEAR // 2, "v_gap").sid.tolist())), dtype=np.int64)
print(f"{len(EV_ALL)} released N-{LEVEL_BASE} scenarios on disk ({time.perf_counter() - t0:.0f} s); {len(ordinary)} drawn at random, {len(near)} nearest to a limit "
      f"(a line within {NR_.line_gap.nsmallest(N_NEAR // 2).max() * 100:.2f} % of its rating or a voltage within {NR_.v_gap.nsmallest(N_NEAR - N_NEAR // 2).max():.4f} p.u. of a limit)")


def our_from(d, start, max_iter=30):
    Y, S, ref, pv, pq = system_of(d)
    V0 = {"flat": np.where(d["PQ"], 1.0, d["Vm"]) * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0))),
          "base": d["Vm"] * np.exp(1j * np.radians(d["Va"]))}[start]
    V, ok, it, sec = klu_solve(Y, S, V0, ref, pv, pq, max_iter, TOL) if HAVE_KLU else (lambda r: (r[0], bool(r[2][-1] <= TOL), r[1], 0.0))(newton(Y, S, V0, pv, pq, tol=TOL, max_it=max_iter))
    return V, ok, it


preload_raw(LOCAL_DIR, LEVEL_BASE, np.r_[ordinary, near])
rows = []
for group, ids in [("random", ordinary), ("near a limit", near)]:
    for sid in ids:
        d = EV_ALL[int(sid)]; ev = ScenarioEval(d)
        bus, br, gen = raw_tables(LOCAL_DIR, LEVEL_BASE, int(sid)); ind = Indep(bus, br, gen)
        rf = ind.solve(start="flat"); rb = ind.solve(start="base", max_it=10)
        Vo, oko, ito = our_from(d, "flat")
        dec_o = LambdaCase(d).decisions(1.0, Vo)
        over_o = set(d["line"][LambdaCase(d).rated][LambdaCase(d).loading(Vo) >= 1.0].tolist())
        rows.append({"group": group, "sid": int(sid), "true_overloads": int(ev.over_t.sum()), "true_vviol": int(ev.vio_t.sum()),
                     "pypower_flat_converged": rf["converged"], "pypower_flat_iterations": rf["iterations"], "pypower_flat_seconds": rf["seconds"],
                     "pypower_vs_released_max_dVm": float(np.abs(rf["Vm"] - d["Vm"]).max()), "pypower_from_released_iterations": rb["iterations"],
                     "ours_flat_converged": bool(oko), "ours_flat_iterations": int(ito), "pypower_vs_ours_max_dVm": float(np.abs(rf["Vm"] - np.abs(Vo)).max()),
                     "overloads_pypower": rf["n_over"], "overloads_ours": dec_o["n_over"], "overload_sets_identical": set(rf["overloaded_idx"]) == over_o,
                     "vviol_pypower": rf["n_low_v"] + rf["n_high_v"], "vviol_ours": dec_o["n_low_v"] + dec_o["n_high_v"],
                     "max_loading_pypower": rf["max_loading"], "max_loading_ours": dec_o["max_loading"],
                     "min_vm_pypower": rf["min_vm"], "min_vm_ours": dec_o["min_vm"]})
C2 = pd.DataFrame(rows)
C2.to_csv(Path(OUT_DIR) / "released_cases.csv", index=False)
agree_over = (C2.overloads_pypower == C2.true_overloads) & C2.overload_sets_identical
agree_v = C2.vviol_pypower == C2.true_vviol
print(f"PYPOWER from a flat start converged on {int(C2.pypower_flat_converged.sum())} of {len(C2)} cases ({C2.pypower_flat_iterations.mean():.1f} iterations, "
      f"{C2.pypower_flat_seconds.median():.2f} s median); its solution is within {C2.pypower_vs_released_max_dVm.max():.1e} p.u. of the released one and "
      f"{C2.pypower_vs_ours_max_dVm.max():.1e} p.u. of ours (largest bus difference over all cases).")
print(f"Decisions: the overload sets of PYPOWER, ours and the released solution coincide in {int(agree_over.sum())} of {len(C2)} cases; voltage violation counts in "
      f"{int(agree_v.sum())} of {len(C2)}. Disagreements, if any, are listed below (a line or bus within solver precision of its limit).")
dis = C2[~agree_over | ~agree_v]
if len(dis):
    display(dis[["group", "sid", "true_overloads", "overloads_pypower", "overloads_ours", "true_vviol", "vviol_pypower", "vviol_ours", "max_loading_pypower", "max_loading_ours"]])
display(C2.groupby("group")[["pypower_vs_released_max_dVm", "pypower_vs_ours_max_dVm", "pypower_flat_iterations", "ours_flat_iterations"]].max())

## 5. The constructed N-20 cases that no classical start solved, and the recovered one

The 479 cases of notebook `clustered_constructions` are rebuilt with the same seed and construction code, and the rebuilt table is compared with
the recorded one (base point, construction and radius of every case). **Notebook `clustered_constructions` stored neither its removed-line lists
nor its base pool, and its construction drew from one random stream over that pool, so the rebuild is exact only when the
fingerprint matches.** When it does not, what follows is a *separate experiment* on a construction of this notebook's own
(the first 120 operating points of the two data blocks downloaded here, same code, same seed), labelled as such in the
outputs; it validates the method on cases of the same kind, not the construction cases themselves. Every case built here is written
with its base point and its removed lines (`replay_cases.csv`), so this notebook's cases can be replayed exactly.
PYPOWER is then started from the pre-contingency state, flat and DC on every case our starts could not solve. For the
recovered case, GENCO runs on the CPU, Newton from its answer is repeated, and the state it reaches is tested under
PYPOWER's equations and for physical plausibility.

In [ ]:
# Four ways of removing EXTRA more lines from an N-2 operating point, every one keeping the grid in one piece
import networkx as nx
d0 = next(iter(BASE.values()))
G = nx.Graph()
for f_, t_ in zip(d0["f"], d0["t"]):
    G.add_edge(int(f_), int(t_))
t0 = time.perf_counter()
EB = nx.edge_betweenness_centrality(G, k=min(BETWEENNESS_SAMPLES, G.number_of_nodes()), seed=SEED, normalized=True)
LINE_BTW = np.array([EB.get((int(f_), int(t_)), EB.get((int(t_), int(f_)), 0.0)) for f_, t_ in zip(d0["f"], d0["t"])])
print(f"edge betweenness of the intact grid ({G.number_of_nodes()} buses, {G.number_of_edges()} bus pairs) in {time.perf_counter() - t0:.0f} s; "
      f"top line carries {100 * LINE_BTW.max():.1f} % of sampled shortest paths")
CONSTRUCTIONS = {"nearest": "the lines nearest a random centre bus (the construction of notebooks 07 and K1)",
                 "uniform": "random lines anywhere (like the released test sets)",
                 "corridor": f"the highest-betweenness lines within {REGION_HOPS} hops of a random centre (a corridor / backbone outage)",
                 "footprint": f"lines within {FOOT_HOPS} hops of a random centre, each removed with probability falling as exp(-hops/{FOOT_SCALE}) (a soft, patchy footprint)"}


def pick_lines(d, extra, mode, rng):
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    centre = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=centre)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    if mode == "nearest":
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    elif mode == "uniform":
        order = rng.permutation(cand)
    elif mode == "corridor":
        h = REGION_HOPS
        while (dist <= h).sum() < 3 * extra and h < 12:
            h += 1
        reg = cand[dist <= h]
        order = reg[np.lexsort((rng.random(len(reg)), -LINE_BTW[reg]))]
    elif mode == "footprint":
        sel = dist <= FOOT_HOPS
        reg, w = cand[sel], np.exp(-dist[sel] / FOOT_SCALE)
        keys = rng.random(len(reg)) ** (1.0 / w)          # weighted sampling without replacement
        order = reg[np.argsort(-keys)]
    else:
        raise ValueError(mode)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, np.nan, centre
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, centre


t0 = time.perf_counter()
rng = np.random.default_rng(SEED + 1)
CASES2, not_built = [], 0
sid = 940_000_000
for b_id in BASE_IDS:
    d = BASE[int(b_id)]
    extra = TOTAL_OUT - int((~d["on"]).sum())
    if extra <= 0:
        continue
    for mode in CONSTRUCTIONS:
        rem, radius, centre = pick_lines(d, extra, mode, rng)
        if rem is None:
            not_built += 1; continue
        d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
        d2["on"][rem] = False
        d2["Va_dc"] = dc_angles(d2)
        case = LambdaCase(d2)
        V1, who = case.solve_any(1.0, [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)], 30, TOL)
        if V1 is not None:
            d2["Vm"], d2["Va"] = np.abs(V1), np.degrees(np.angle(V1))
        sid += 1
        CASES2.append({"sid": sid, "base": int(b_id), "mode": mode, "radius_hops": radius, "centre": centre, "rows": rem,
                       "removed_lines": d["line"][rem].astype(int).tolist(), "d": d2, "case": case, "V1": V1, "solvable": V1 is not None,
                       "solved_from": who, "btw_sum": float(LINE_BTW[rem].sum())})
C2_N2 = pd.DataFrame([{k: c[k] for k in ["sid", "base", "mode", "radius_hops", "centre", "solvable", "solved_from", "btw_sum"]} for c in CASES2])
print(f"{len(CASES2)} cases built in {time.perf_counter() - t0:.0f} s (could not build {not_built})")
display(C2_N2.groupby("mode").agg(cases=("sid", "count"), radius_hops=("radius_hops", "median"), no_AC_solution=("solvable", lambda x: int((~x).sum())),
                               betweenness_of_removed_lines=("btw_sum", "median")))

In [ ]:
# Did the replay rebuild exactly the 479 cases of notebook `clustered_constructions`? (same base points, same seed, same construction code)
FP = pd.DataFrame(PLAN["n2_fingerprint"], columns=["sid", "base", "mode", "radius_hops", "solvable"])
C2r = C2_N2.merge(FP, on="sid", how="outer", suffixes=("_now", "_N2"), indicator=True)
same = (C2r["_merge"] == "both") & (C2r.base_now == C2r.base_N2) & (C2r.mode_now == C2r.mode_N2) & (np.abs(C2r.radius_hops_now - C2r.radius_hops_N2) < 1e-9)
same_solv = same & (C2r.solvable_now == C2r.solvable_N2)
N2_REPLAY_OK = bool(same.all()) and len(C2r) == len(FP)
print(f"Replay: {int(same.sum())} of {len(FP)} constructed cases rebuilt with the same base point, construction and radius; {int(same_solv.sum())} also with the same "
      f"solved/unresolved outcome of the classical starts." + ("" if N2_REPLAY_OK else " THE REPLAY DIFFERS: the cases below are a SEPARATE CONSTRUCTION of this "
      "notebook (same code and seed on its own base pool), not the N2 cases; they are labelled so in the outputs."))
# every case built here, with its base point and removed lines, so that this notebook's cases can be replayed exactly
pd.DataFrame([{"sid": c["sid"], "base": int(c["base"]), "mode": c["mode"], "radius_hops": c["radius_hops"], "solvable": bool(c["solvable"]),
               "removed_lines": json.dumps([int(x) for x in c["removed_lines"]]), "identical_to_N2_case": bool(same_solv[C2r.sid == c["sid"]].all()) if (C2r.sid == c["sid"]).any() else False}
              for c in CASES2]).to_csv(Path(OUT_DIR) / "replay_cases.csv", index=False)
print("written replay_cases.csv (base point and removed lines of every case built here)")
if not same_solv.all():
    display(C2r[~same_solv][["sid", "base_now", "mode_now", "solvable_now", "solvable_N2"]].head(20))

In [ ]:
# Every construction case unresolved by our classical starts: does PYPOWER, from the same three starts, find a solution?

t0 = time.perf_counter()
rows = []
UNRES = [c for c in CASES2 if not c["solvable"]]
IND_BASE = {}
for c in UNRES:
    b_ = int(c["base"])
    if b_ not in IND_BASE:
        IND_BASE[b_] = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, b_))
    ind = IND_BASE[b_]
    rem = c["removed_lines"]
    r = {"sid": c["sid"], "base": b_, "mode": c["mode"], "removed_lines": json.dumps([int(x) for x in rem]), "construction": "N2 case" if N2_REPLAY_OK else "V1's own (separate experiment)", "ours_solved": False}
    found = []
    for st in ["base", "flat", "dc"]:
        x = ind.solve(removed_idx=rem, lam=1.0, start=st, max_it=MAX_NR)
        r[f"pypower_{st}_converged"] = x["converged"]; r[f"pypower_{st}_iterations"] = x["iterations"]
        if x["converged"]:
            found.append((st, x))
    r["pypower_any_converged"] = bool(found)
    if found:
        st, x = found[0]
        comp = complementarity(ind.bus, ind.gen, x["Vm"], x["Qg_bus_Mvar"])
        r.update({"found_from": st, "min_vm": x["min_vm"], "n_low_v": x["n_low_v"], "n_over": x["n_over"], "max_loading": x["max_loading"],
                  "buses_below_0.8": int((x["Vm"] < 0.8).sum()), "gens_beyond_q_limits": comp["beyond_limits"],
                  "max_dVm_from_pre_contingency": float(np.abs(x["Vm"] - ind.bus.Vm.to_numpy()).max())})
    rows.append(r)
U3 = pd.DataFrame(rows)
U3.to_csv(Path(OUT_DIR) / "unresolved_n2_cases.csv", index=False)
print(f"{len(U3)} unresolved constructed cases, PYPOWER from the pre-contingency state, flat and DC starts ({MAX_NR} iterations each) in {time.perf_counter() - t0:.0f} s: "
      f"converged on {int(U3.pypower_any_converged.sum())} (from the pre-contingency state {int(U3.pypower_base_converged.sum())}, flat {int(U3.pypower_flat_converged.sum())}, "
      f"DC {int(U3.pypower_dc_converged.sum())}).")
if U3.pypower_any_converged.any():
    print("Cases PYPOWER solved although our starts did not (with the plausibility of the state it found):")
    display(U3[U3.pypower_any_converged][["sid", "mode", "found_from", "min_vm", "buses_below_0.8", "n_low_v", "n_over", "gens_beyond_q_limits", "max_dVm_from_pre_contingency"]])

In [ ]:
# The one construction case on which Newton from GENCO's answer converged (20 iterations) although no classical start did: validated independently

c = next((c for c in CASES2 if c["sid"] == PLAN["recovered_sid"]), None)
if c is None:
    print("the recovered case is not among the rebuilt cases (replay differs); skipped")
    REC = {"available": False}
else:
    d2, case = c["d"], c["case"]
    ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, int(c["base"]))); rem = c["removed_lines"]
    t0 = time.perf_counter()
    vm, va = run_genco([storm_graph(c)])[0]
    Vg = case.state(vm, np.degrees(va))
    Vk, okk, itk, sk = case.solve(1.0, Vg, 30, TOL)
    Vk100, okk100, itk100, _ = case.solve(1.0, Vg, 100, TOL)
    g_dec = case.decisions(1.0, Vg)
    REC = {"available": True, "sid": c["sid"], "base": int(c["base"]), "mode": c["mode"], "removed_lines": rem, "ours_solved_by_classical_starts": bool(c["solvable"]),
           "genco_min_vm": g_dec["min_vm"], "genco_n_low_v": g_dec["n_low_v"], "genco_n_over": g_dec["n_over"],
           "ours_from_genco_converged_30": bool(okk), "ours_from_genco_iterations_30": int(itk), "ours_from_genco_converged_100": bool(okk100), "ours_from_genco_iterations_100": int(itk100)}
    ppc = ind.ppc(rem, 1.0)
    if okk:
        Vm_k, Va_k = np.abs(Vk), np.degrees(np.angle(Vk))
        dec_k = case.decisions(1.0, Vk)
        ver = ind.solve(removed_idx=rem, lam=1.0, start=(Vm_k, Va_k), max_it=10)
        comp = complementarity(ind.bus, ind.gen, ver["Vm"], ver["Qg_bus_Mvar"])
        REC.update({"our_state_mismatch_under_pypower_pu": ind.mismatch(ppc, Vk), "pypower_from_our_state_iterations": ver["iterations"],
                    "pypower_from_our_state_max_dVm": float(np.abs(ver["Vm"] - Vm_k).max()), "pypower_from_our_state_converged": ver["converged"],
                    "state_min_vm_pq": dec_k["min_vm"], "state_min_vm_all_buses": float(Vm_k.min()), "state_buses_below_0.8": int((Vm_k < 0.8).sum()),
                    "state_buses_below_0.9": int((Vm_k < 0.9).sum()), "state_n_low_v": dec_k["n_low_v"], "state_n_high_v": dec_k["n_high_v"],
                    "state_n_over": dec_k["n_over"], "state_max_loading": dec_k["max_loading"],
                    "pypower_metrics_on_state": {"n_over": ver["n_over"], "n_low_v": ver["n_low_v"], "n_high_v": ver["n_high_v"], "max_loading": ver["max_loading"], "min_vm": ver["min_vm"]},
                    "state_max_dVm_from_pre_contingency": float(np.abs(Vm_k - BASE[int(c["base"])]["Vm"]).max()),
                    "state_gens_beyond_q_limits": comp["beyond_limits"], "state_pv_buses_with_limits": comp["pv_buses_with_limits"]})
    starts = {"genco": (np.abs(Vg), np.degrees(np.angle(Vg))), "base": "base", "flat": "flat", "dc": "dc"}
    for name, st in starts.items():
        for mi in [30, 100]:
            x = ind.solve(removed_idx=rem, lam=1.0, start=st, max_it=mi)
            REC[f"pypower_from_{name}_{mi}"] = {"converged": x["converged"], "iterations": x["iterations"], "min_vm": x["min_vm"] if x["converged"] else None,
                                                "max_dVm_vs_our_state": float(np.abs(x["Vm"] - np.abs(Vk)).max()) if (x["converged"] and okk) else None}
    for name, V0 in [("base", case.state(BASE[int(c["base"])]["Vm"], BASE[int(c["base"])]["Va"])), ("flat", case.flat), ("dc", case.dc)]:
        V_, ok_, it_, _ = case.solve(1.0, V0, 100, TOL)
        REC[f"ours_from_{name}_100"] = {"converged": bool(ok_), "iterations": int(it_), "max_dVm_vs_state_from_genco": float(np.abs(np.abs(V_) - np.abs(Vk)).max()) if (ok_ and okk) else None}
    REC["seconds"] = time.perf_counter() - t0
    json.dump(REC, open(Path(OUT_DIR) / "recovered_case.json", "w"), indent=1, default=float)
    print(f"Case {c['sid']} ({c['mode']}, base {c['base']}): GENCO shows a lowest voltage of {g_dec['min_vm']:.3f} p.u., {g_dec['n_low_v']} buses below the limit, {g_dec['n_over']} overloads.")
    print(f"Newton (lightsim2grid) from GENCO's answer: converged {okk} in {itk} iterations (cap 30); with a cap of 100: {okk100} in {itk100}.")
    if okk:
        print(f"That state under PYPOWER's independently built equations: mismatch {REC['our_state_mismatch_under_pypower_pu']:.1e} p.u.; PYPOWER started there converged in "
              f"{REC['pypower_from_our_state_iterations']} iteration(s) moving the voltages by {REC['pypower_from_our_state_max_dVm']:.1e} p.u. -> it is a solution of the equations.")
        print(f"Its plausibility: lowest voltage {REC['state_min_vm_all_buses']:.3f} p.u. ({REC['state_buses_below_0.8']} buses below 0.8, {REC['state_buses_below_0.9']} below 0.9), "
              f"{REC['state_n_over']} overloads, largest loading {REC['state_max_loading']:.2f}, {REC['state_gens_beyond_q_limits']} of {REC['state_pv_buses_with_limits']} limited PV buses "
              f"beyond their reactive limits, largest voltage change from the pre-contingency state {REC['state_max_dVm_from_pre_contingency']:.3f} p.u.")
    for name in starts:
        a, b = REC[f"pypower_from_{name}_30"], REC[f"pypower_from_{name}_100"]
        print(f"PYPOWER from {name:5s}: 30 iterations -> {a['converged']} ({a['iterations']}); 100 iterations -> {b['converged']} ({b['iterations']})"
              + (f", same state as ours to {b['max_dVm_vs_our_state']:.1e} p.u." if b["max_dVm_vs_our_state"] is not None else ""))
    for name in ["base", "flat", "dc"]:
        a = REC[f"ours_from_{name}_100"]
        print(f"ours (lightsim2grid) from {name:5s} with 100 iterations: {a['converged']} ({a['iterations']})" + (f", same state to {a['max_dVm_vs_state_from_genco']:.1e} p.u." if a["max_dVm_vs_state_from_genco"] is not None else ""))

## 6. Tracked-branch folds and the identity of the tracked branch

For each storm, at ten load factors around its fold: our three classical starts and our continuation state, and
PYPOWER from flat, DC, the pre-contingency state and the continuation state. Distinct converged states are clustered;
the continuation state should be the highest-voltage solution (the operating branch); beyond the fold no start should
converge.

In [ ]:
# Tracked-branch folds: which branch is tracked below the fold, and does PYPOWER see the same fold?

def storm_dict(st):
    d = BASE[int(st["base"])]
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][np.isin(d2["line"], np.asarray(st["removed_lines"], dtype=int))] = False
    d2["Va_dc"] = dc_angles(d2)
    return d2


def classical_starts(case, d):
    return [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)]


def cluster_states(states, tol=1e-4):
    """Distinct solutions among (name, Vm) pairs: groups whose voltages agree to tol; sorted by lowest voltage, highest first."""
    groups = []
    for name, Vm in states:
        for g_ in groups:
            if np.max(np.abs(g_["Vm"] - Vm)) < tol:
                g_["members"].append(name); break
        else:
            groups.append({"Vm": np.asarray(Vm), "members": [name]})
    for g_ in groups:
        g_["min_vm"] = float(g_["Vm"].min())
    return sorted(groups, key=lambda g_: -g_["min_vm"])


REL = [-0.15, -0.10, -0.05, -0.02, -0.005, 0.0, 0.0025, 0.005, 0.01, 0.05]
t0 = time.perf_counter()
rows, fold_summary = [], []
for st in PLAN["fold_storms"]:
    d2 = storm_dict(st); case = LambdaCase(d2); d0 = BASE[int(st["base"])]
    ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, int(st["base"]))); rem = st["removed_lines"]
    # our solution at the fold: the stored lam_max, stepping down a little if the bisection grid does not reproduce exactly
    lam_star, V_star = None, None
    for lam in [st["lam_max"], st["lam_max"] - 0.0025, st["lam_max"] - 0.005, st["lam_max"] - 0.01]:
        V_, who = case.solve_any(lam, classical_starts(case, d0), 30, TOL)
        if V_ is not None:
            lam_star, V_star, who_star = lam, V_, who; break
    if V_star is None:
        nose = find_lambda_max(case, None if not st["solvable_at_1"] else case.solve_any(1.0, classical_starts(case, d0), 30, TOL)[0], tol=TOL)
        lam_star, V_star, who_star = nose["lam_max"], nose["V_max"], "find_lambda_max"
    lams = [round(lam_star + r_, 6) for r_ in REL]
    cont = truth_below_nose(case, lam_star, V_star, [l_ for l_ in lams if l_ <= lam_star], tol=TOL)
    n_two, upper_ok, beyond_conv = 0, True, 0
    for lam in lams:
        beyond = lam > lam_star + 1e-9
        states, rec = [], {"sid": st["sid"], "rising_in_K1": st["rising"], "solvable_at_1": st["solvable_at_1"], "lam": lam, "lam_rel": round(lam - lam_star, 6), "beyond_fold": beyond}
        for name, V0 in classical_starts(case, d0):
            V_, ok_, it_, _ = case.solve(lam, V0, 30, TOL)
            ok_ = bool(ok_ and np.all(np.isfinite(V_)) and np.abs(V_).min() > 0.2)
            rec[f"ours_{name}"] = ok_
            if ok_:
                states.append((f"ours_{name}", np.abs(V_)))
        if lam in cont:
            states.append(("ours_continuation", np.abs(cont[lam])))
            rec["ours_continuation"] = True
            rec["continuation_min_vm"] = float(np.abs(cont[lam]).min())
            rec["continuation_mismatch_pypower_pu"] = ind.mismatch(ind.ppc(rem, lam), cont[lam])
        else:
            rec["ours_continuation"] = False
        pstarts = [("base", "base"), ("flat", "flat"), ("dc", "dc")]
        if lam in cont:
            pstarts.append(("continuation", (np.abs(cont[lam]), np.degrees(np.angle(cont[lam])))))
        elif V_star is not None:
            pstarts.append(("fold_state", (np.abs(V_star), np.degrees(np.angle(V_star)))))
        for name, s0 in pstarts:
            x = ind.solve(removed_idx=rem, lam=lam, start=s0, max_it=30)
            rec[f"pypower_{name}"] = x["converged"]
            if name == "continuation":
                rec["pypower_from_continuation_iterations"] = x["iterations"]; rec["pypower_from_continuation_max_dVm"] = float(np.abs(x["Vm"] - np.abs(cont[lam])).max())
            if x["converged"]:
                states.append((f"pypower_{name}", x["Vm"]))
        groups = cluster_states(states)
        rec["distinct_solutions"] = len(groups)
        rec["solutions_min_vm"] = [round(g_["min_vm"], 4) for g_ in groups]
        rec["solution_members"] = [",".join(g_["members"]) for g_ in groups]
        if len(groups) > 1:
            n_two += 1
        if lam in cont and groups and "ours_continuation" not in groups[0]["members"]:
            upper_ok = False
        if beyond and groups:
            beyond_conv += 1
        rows.append(rec)
    fold_summary.append({"sid": st["sid"], "rising_in_K1": st["rising"], "solvable_at_1": st["solvable_at_1"], "lam_max_K1": st["lam_max"], "lam_fold_K2": st["lam_fold"],
                         "lam_used": lam_star, "found_from": who_star, "points_with_two_solutions": n_two, "continuation_is_highest_voltage_solution": upper_ok,
                         "points_beyond_fold_where_any_solver_converged": beyond_conv,
                         "pypower_verifies_continuation_states": bool(all(r_["pypower_from_continuation_max_dVm"] < 1e-6 for r_ in rows if r_["sid"] == st["sid"] and "pypower_from_continuation_max_dVm" in r_))})
    print(f"storm {st['sid']}: fold at {lam_star} ({who_star}); points with two distinct solutions {n_two}; continuation state is the highest-voltage solution: {upper_ok}; "
          f"beyond the fold a solver converged at {beyond_conv} of {sum(1 for l_ in lams if l_ > lam_star + 1e-9)} points ({time.perf_counter() - t0:.0f} s)")
F4 = pd.DataFrame(rows); F4.to_csv(Path(OUT_DIR) / "folds_points.csv", index=False)
FS = pd.DataFrame(fold_summary); FS.to_csv(Path(OUT_DIR) / "folds_summary.csv", index=False)
display(FS)
print("Points where more than one solution was found (the highest-voltage solution first; members: which start of which solver reached it):")
display(F4[F4.distinct_solutions > 1][["sid", "lam_rel", "solutions_min_vm", "solution_members", "continuation_min_vm"]])

## 7. Reactive limits: our switching loop against PYPOWER

Three ways of enforcing the limits on the same cases: our loop (limits summed per bus, a bus released when its voltage
recovers past the set-point, slack unlimited), the same convention around PYPOWER's Newton-Raphson, and the MATPOWER
convention (per generator, never released; with the slack unlimited or limited). The complementarity conditions of every
final state are checked: a bus within its limits sits at its set-point, a bus at Qmax at or below it, a bus at Qmin at
or above it.

In [ ]:
# Reactive limits: our switching loop against PYPOWER with the MATPOWER convention and with our convention

rows = []
t0 = time.perf_counter()
for st in PLAN["qlim_storms"]:
    d2 = storm_dict(st); d0 = BASE[int(st["base"])]
    gens = GEN_T[GEN_T.scenario == int(st["base"])].reset_index(drop=True)
    qmin, qmax = bus_q_limits(d2, gens)
    qc = QLimCase(d2, qmin, qmax)
    ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, int(st["base"]))); rem = st["removed_lines"]
    r = {"sid": st["sid"], "N1_solvable_no_limits": st["solvable_no_limits"], "N1_solvable_with_limits": st["solvable_with_limits"], "N1_generators_at_a_limit": st["generators_at_a_limit"]}
    ours = {}
    for name, V0 in [("base", qc.state(d0["Vm"], d0["Va"])), ("flat", qc.flat)]:
        V_, ok_, it_, _ = qc.solve(1.0, V0, 30, TOL)
        held = dict(qc.last["switched"])
        Qg = q_generation(qc.Y, V_, qc.Qd) * SN_MVA
        comp = complementarity(ind.bus, ind.gen, np.abs(V_), Qg) if ok_ else None
        ours[name] = (V_, ok_, held)
        r.update({f"ours_{name}_converged": bool(ok_), f"ours_{name}_rounds": qc.last["outer"], f"ours_{name}_buses_at_limit": len(held),
                  f"ours_{name}_complementarity_violations": comp["complementarity_violations"] if comp else None})
    pyp = {}
    for conv, slack_free, tag in [("release", True, "release"), ("matpower", True, "matpower"), ("matpower", False, "matpower_slack_limited")]:
        for name in ["base", "flat"]:
            x = ind.solve_qlim(removed_idx=rem, lam=1.0, start=name, max_it=30, convention=conv, slack_unlimited=slack_free)
            comp = complementarity(ind.bus, ind.gen, x["Vm"], x["Qg_bus_Mvar"]) if x["converged"] else None
            pyp[(tag, name)] = x
            r.update({f"pypower_{tag}_{name}_converged": x["converged"], f"pypower_{tag}_{name}_rounds": x["rounds"], f"pypower_{tag}_{name}_buses_at_limit": len(x["buses_at_limit"]),
                      f"pypower_{tag}_{name}_complementarity_violations": comp["complementarity_violations"] if comp else None})
    # agreement between our limited solution and PYPOWER's with the same convention, from the same start
    for name in ["base", "flat"]:
        V_, ok_, held = ours[name]; x = pyp[("release", name)]
        if ok_ and x["converged"]:
            r[f"release_{name}_max_dVm"] = float(np.abs(np.abs(V_) - x["Vm"]).max())
            r[f"release_{name}_same_buses_at_limit"] = set(held) == set(x["buses_at_limit"])
        y = pyp[("matpower", name)]
        if ok_ and y["converged"]:
            r[f"matpower_{name}_max_dVm"] = float(np.abs(np.abs(V_) - y["Vm"]).max())
            r[f"matpower_{name}_buses_at_limit_common"] = len(set(held) & set(y["buses_at_limit"]))
    r["pv_buses_with_several_limited_gens"] = int(((np.bincount(gens[(gens.in_service > 0.5) & (gens.max_q_mvar >= gens.min_q_mvar)].bus.astype(int), minlength=d2["nb"]) > 1) & (d2["PV"] | d2["REF"])).sum())
    rows.append(r)
    print(f"storm {st['sid']}: ours from base {r['ours_base_converged']} ({r['ours_base_buses_at_limit']} buses at a limit) | PYPOWER release-convention {r['pypower_release_base_converged']} "
          f"({r['pypower_release_base_buses_at_limit']}), MATPOWER convention {r['pypower_matpower_base_converged']} ({r['pypower_matpower_base_buses_at_limit']}), "
          f"slack limited too {r['pypower_matpower_slack_limited_base_converged']} ({time.perf_counter() - t0:.0f} s)")
Q5 = pd.DataFrame(rows); Q5.to_csv(Path(OUT_DIR) / "qlimits.csv", index=False)
cols = ["sid", "N1_solvable_with_limits", "ours_base_converged", "ours_base_buses_at_limit", "ours_base_complementarity_violations",
        "pypower_release_base_converged", "release_base_max_dVm", "release_base_same_buses_at_limit",
        "pypower_matpower_base_converged", "pypower_matpower_base_buses_at_limit", "pypower_matpower_base_complementarity_violations", "matpower_base_max_dVm",
        "pypower_matpower_slack_limited_base_converged", "pv_buses_with_several_limited_gens"]
display(Q5[[c_ for c_ in cols if c_ in Q5.columns]])

## 8. Where our methods disagreed: the hard case and the points below a fold where Newton from GENCO failed

In [ ]:
# The hard case the screening check accepted with a miss (N-20 scenario 2749): every state under PYPOWER

def adaptive(ev, vm, va_deg):
    tr = newton_trace(ev.Y, ev.Sspec, ev.state(vm, va_deg), ev.pv, ev.pq, tol=TOL, max_it=SCREEN_MAX)
    j = next((j for j in range(1, len(tr)) if tr[j][1] <= TOL_SCREEN), None)
    V = tr[j][0] if j is not None else tr[-1][0]
    return (j if j is not None else SCREEN_MAX), ev.metrics(np.abs(V), np.degrees(np.angle(V))), V, j is not None


HL, HS_ = PLAN["hard"]["level"], PLAN["hard"]["scenario"]
HARD_OUT = {"level": HL, "scenario": HS_}
try:
    on_disk = set(scenario_ids(find_local(LOCAL_DIR, HL, "bus_data")).tolist())
except FileNotFoundError:
    on_disk = set()
if HS_ not in on_disk:
    print(f"scenario {HS_} of N-{HL} is not on disk; skipped"); HARD_OUT["available"] = False
else:
    HARD_OUT["available"] = True
    ids = np.array([HS_], dtype=np.int64)
    d = truth_dicts(HL, ids)[HS_]; ev = ScenarioEval(d)
    bus, br, gen = raw_tables(LOCAL_DIR, HL, HS_); ind = Indep(bus, br, gen)

    def released_graphs(level, ids):
        bus_ = read_all(find_local(LOCAL_DIR, level, "bus_data"), ids); gen_ = read_all(find_local(LOCAL_DIR, level, "gen_data"), ids)
        br_ = read_all(find_local(LOCAL_DIR, level, "branch_data"), ids)
        bus_ = merge_agg_gen_into_bus(bus_, gen_)
        bg_, gg_, rg_ = bus_.groupby("scenario"), gen_.groupby("scenario"), br_.groupby("scenario")
        gs = []
        for s_ in ids:
            b_ = bg_.get_group(int(s_)).sort_values("bus").reset_index(drop=True)
            data = build_hetero_data_for_scenario(int(s_), b_, gg_.get_group(int(s_)), rg_.get_group(int(s_)))
            NORMALIZER.transform(data)
            gs.append(TRANSFORM(data))
        return gs

    g = released_graphs(HL, ids)
    vm, va = run_genco(g)[0]
    Vg = ev.state(vm, np.degrees(va))
    j, mg, Vs, passed = adaptive(ev, vm, np.degrees(va))
    Y, S, ref, pv, pq = system_of(d)
    Vc, okc, itc, _ = klu_solve(Y, S, Vg, ref, pv, pq, 30, TOL) if HAVE_KLU else (lambda r_: (r_[0], bool(r_[2][-1] <= TOL), r_[1], 0.0))(newton(Y, S, Vg, pv, pq, tol=TOL, max_it=30))
    mc = ev.metrics(np.abs(Vc), np.degrees(np.angle(Vc)))
    states = {"released": (d["Vm"], d["Va"]), "genco": (np.abs(Vg), np.degrees(np.angle(Vg))), "screening_check": (np.abs(Vs), np.degrees(np.angle(Vs))), "converged_state": (np.abs(Vc), np.degrees(np.angle(Vc)))}
    for name, (Vm_, Va_) in states.items():
        x = ind.solve(start=(Vm_, Va_), max_it=30)
        m_ = ev.metrics(Vm_, Va_)
        HARD_OUT[name] = {"mismatch_pypower_pu": ind.mismatch(ind.ppc(), Vm_ * np.exp(1j * np.radians(Va_))), "pypower_iterations_from_here": x["iterations"],
                          "pypower_converged_from_here": x["converged"], "pypower_max_dVm_from_here": float(np.abs(x["Vm"] - Vm_).max()),
                          "missed_overloads_vs_released": m_["missed_overloads"], "missed_vviol_vs_released": m_["missed_vviol"], "err_vm_max": m_["err_vm_max"]}
    xf = ind.solve(start="flat")
    HARD_OUT.update({"screening_steps": int(j), "screening_passed": bool(passed), "converged_from_genco": bool(okc), "iterations_from_genco": int(itc),
                     "pypower_flat_converged": xf["converged"], "pypower_flat_vs_released_max_dVm": float(np.abs(xf["Vm"] - d["Vm"]).max()),
                     "true_overloads": int(ev.over_t.sum()), "true_vviol": int(ev.vio_t.sum()),
                     "pypower_overloads": xf["n_over"], "pypower_vviol": xf["n_low_v"] + xf["n_high_v"]})
    miss_bus = np.nonzero(ev.vio_t & ~(d["PQ"] & ((np.abs(Vs) < d["min_vm_pu"]) | (np.abs(Vs) > d["max_vm_pu"]))))[0]
    HARD_OUT["missed_buses_after_screening"] = [{"bus": int(b_), "limit": [float(d["min_vm_pu"][b_]), float(d["max_vm_pu"][b_])], "released_vm": float(d["Vm"][b_]),
                                                 "pypower_vm": float(xf["Vm"][b_]), "genco_vm": float(np.abs(Vg)[b_]), "screening_vm": float(np.abs(Vs)[b_]),
                                                 "converged_vm": float(np.abs(Vc)[b_])} for b_ in miss_bus]
    json.dump(HARD_OUT, open(Path(OUT_DIR) / "hard_case.json", "w"), indent=1, default=float)
    print(f"N-{HL} scenario {HS_}: {HARD_OUT['true_overloads']} true overloads, {HARD_OUT['true_vviol']} voltage violations (released solution); PYPOWER from flat: "
          f"{HARD_OUT['pypower_overloads']} / {HARD_OUT['pypower_vviol']}, within {HARD_OUT['pypower_flat_vs_released_max_dVm']:.1e} p.u. of the released state.")
    for name in states:
        h = HARD_OUT[name]
        print(f"  {name:22s}: mismatch under PYPOWER {h['mismatch_pypower_pu']:.1e} p.u.; PYPOWER from here converges in {h['pypower_iterations_from_here']} it. moving {h['pypower_max_dVm_from_here']:.1e} p.u.; "
              f"misses vs released: {h['missed_overloads_vs_released']} overloads, {h['missed_vviol_vs_released']} voltage violations")
    print(f"  screening check: {j} steps, passed {passed}; Newton to convergence from GENCO: {okc} in {itc} iterations.")
    if HARD_OUT["missed_buses_after_screening"]:
        display(pd.DataFrame(HARD_OUT["missed_buses_after_screening"]))

In [ ]:
# Points below a fold where Newton from GENCO's answer did not converge in storm: the same points under PYPOWER

t0 = time.perf_counter()
by_storm = {}
for p_ in PLAN["failed_points"]:
    by_storm.setdefault(p_["sid"], []).append(p_)
rows = []
for sid, pts in by_storm.items():
    st = pts[0]; d2 = storm_dict(st); case = LambdaCase(d2); d0 = BASE[int(st["base"])]
    ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, int(st["base"]))); rem = st["removed_lines"]
    g0 = build_graph(sid, int(st["base"]), removed_lines=rem)
    outs = run_genco([scaled_graph(g0, p_["lam"]) for p_ in pts])
    # the tracked solutions: march down from the fold (as storm did)
    lam_star, V_star = None, None
    for lam in [st["lam_max"], st["lam_max"] - 0.0025, st["lam_max"] - 0.005]:
        V_, who = case.solve_any(lam, classical_starts(case, d0), 30, TOL)
        if V_ is not None:
            lam_star, V_star = lam, V_; break
    cont = truth_below_nose(case, lam_star, V_star, [p_["lam"] for p_ in pts if p_["lam"] <= lam_star], tol=TOL) if V_star is not None else {}
    for p_, (vm, va) in zip(pts, outs):
        lam = p_["lam"]
        Vg = case.state(vm, np.degrees(va))
        V_, ok30, it30, _ = case.solve(lam, Vg, 30, TOL)
        V100, ok100, it100, _ = case.solve(lam, Vg, 100, TOL)
        rec = {"sid": sid, "lam": lam, "lam_rel_K1": p_["lam_rel"], "K1_true_min_vm": p_["true_min_vm"], "K1_flat_converged": p_["flat_converged"], "K1_dc_converged": p_["dc_converged"],
               "ours_from_genco_30": bool(ok30 and np.abs(V_).min() > 0.2), "ours_from_genco_100": bool(ok100 and np.abs(V100).min() > 0.2), "ours_from_genco_100_iterations": int(it100)}
        states = []
        if rec["ours_from_genco_100"]:
            states.append(("ours_genco_100", np.abs(V100)))
        for name, V0 in classical_starts(case, d0):
            Vx, okx, _, _ = case.solve(lam, V0, 30, TOL)
            okx = bool(okx and np.all(np.isfinite(Vx)) and np.abs(Vx).min() > 0.2)
            rec[f"ours_{name}"] = okx
            if okx:
                states.append((f"ours_{name}", np.abs(Vx)))
        if lam in cont:
            states.append(("ours_continuation", np.abs(cont[lam]))); rec["continuation_min_vm"] = float(np.abs(cont[lam]).min())
            x = ind.solve(removed_idx=rem, lam=lam, start=(np.abs(cont[lam]), np.degrees(np.angle(cont[lam]))), max_it=10)
            rec["pypower_verifies_continuation"] = bool(x["converged"] and np.abs(x["Vm"] - np.abs(cont[lam])).max() < 1e-6)
        for name, s0 in [("genco", (np.abs(Vg), np.degrees(np.angle(Vg)))), ("base", "base"), ("flat", "flat"), ("dc", "dc")]:
            for mi in ([30, 100] if name == "genco" else [30]):
                x = ind.solve(removed_idx=rem, lam=lam, start=s0, max_it=mi)
                rec[f"pypower_{name}_{mi}"] = x["converged"]
                if x["converged"]:
                    states.append((f"pypower_{name}_{mi}", x["Vm"]))
        groups = cluster_states(states)
        rec["distinct_solutions"] = len(groups); rec["solutions_min_vm"] = [round(g_["min_vm"], 4) for g_ in groups]; rec["solution_members"] = [",".join(g_["members"]) for g_ in groups]
        rec["genco_min_vm"] = float(np.abs(Vg)[d2["PQ"]].min())
        rows.append(rec)
    print(f"storm {sid}: {len(pts)} points done ({time.perf_counter() - t0:.0f} s)")
P6 = pd.DataFrame(rows); P6.to_csv(Path(OUT_DIR) / "failed_points.csv", index=False)
print(f"{len(P6)} points. Newton from GENCO's answer: ours converged at {int(P6.ours_from_genco_30.sum())} (30 it.) / {int(P6.ours_from_genco_100.sum())} (100 it.); "
      f"PYPOWER from the same GENCO state at {int(P6.pypower_genco_30.sum())} (30 it.) / {int(P6.pypower_genco_100.sum())} (100 it.). "
      f"PYPOWER verified the tracked solution at {int(P6.get('pypower_verifies_continuation', pd.Series(dtype=bool)).fillna(False).sum())} of {int(P6.get('pypower_verifies_continuation', pd.Series(dtype=bool)).notna().sum())} points. "
      f"Points with two distinct solutions: {int((P6.distinct_solutions > 1).sum())}.")
display(P6[["sid", "lam_rel_K1", "K1_true_min_vm", "genco_min_vm", "ours_from_genco_100", "pypower_genco_100", "ours_base", "ours_flat", "ours_dc", "pypower_base_30", "pypower_flat_30", "pypower_dc_30", "distinct_solutions", "solutions_min_vm"]])

## 9. Summary and output files

In [ ]:
import json, platform
summary = {
    "provenance": PROVENANCE,
    "graphkit": {"checked_out_commit": subprocess.run(["git", "-C", GRAPHKIT_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
                 "module_file": os.path.abspath(gridfm_graphkit.__file__), "branch": GRAPHKIT_BRANCH},
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__,
                 "pypower": _pkg_version("PYPOWER"), "lightsim2grid": HAVE_KLU, "device": str(DEVICE), "torch_scatter_shim": bool(USED_SHIM)},
    "config": {k: v for k, v in globals().items() if k in ["REPO_ID", "LEVEL_BASE", "BASE_PARTS", "N_BASE", "SEED", "TOL", "TOL_SCREEN", "MAX_NR", "N_ORDINARY", "N_NEAR", "TOTAL_OUT", "REGION_HOPS", "FOOT_HOPS", "FOOT_SCALE", "BETWEENNESS_SAMPLES"]},
    "conventions": {"stored_vs_recomputed_admittance_max_rel_diff": float(P1.max_rel_diff_Y.max()), "released_mismatch_under_pypower_pu_max": float(P1.released_mismatch_pypower_pu.max()),
                    "gen_sum_vs_bus_Pg_max_diff_MW": float(P1.gen_sum_vs_bus_Pg_max_diff_MW.max()), "gen_sum_vs_bus_Qg_max_diff_Mvar": float(P1.gen_sum_vs_bus_Qg_max_diff_Mvar.max()),
                    "released_mismatch_if_GS_BS_were_MW_pu_max": float(P1.released_mismatch_if_GS_BS_were_MW_pu.max()),
                    "pv_or_ref_without_gen_max": int(P1.pv_or_ref_without_gen.max()), "pq_with_in_service_gen_max": int(P1.pq_with_in_service_gen.max()),
                    "pv_buses_with_several_limited_gens_median": float(P1.pv_buses_with_several_limited_gens.median()),
                    "pypower_from_released_max_dVm": float(M1.max_dVm_vs_released.max()) if len(M1) else None},
    "released_cases": {"cases": int(len(C2)), "pypower_flat_converged": int(C2.pypower_flat_converged.sum()), "pypower_vs_released_max_dVm": float(C2.pypower_vs_released_max_dVm.max()),
                       "pypower_vs_ours_max_dVm": float(C2.pypower_vs_ours_max_dVm.max()), "overload_sets_agree": int(agree_over.sum()), "vviol_counts_agree": int(agree_v.sum())},
    "n2_replay_identical": N2_REPLAY_OK,
    "unresolved_n2_cases": {"cases": int(len(U3)), "pypower_converged_any_start": int(U3.pypower_any_converged.sum()),
                            "pypower_converged_sids": U3[U3.pypower_any_converged].sid.tolist()},
    "recovered_case": REC,
    "folds": FS.to_dict(orient="records"),
    "qlimits": Q5.to_dict(orient="records"),
    "hard_case": HARD_OUT,
    "failed_points": {"points": int(len(P6)), "ours_from_genco_100": int(P6.ours_from_genco_100.sum()), "pypower_from_genco_100": int(P6.pypower_genco_100.sum()),
                      "points_with_two_solutions": int((P6.distinct_solutions > 1).sum()),
                      "pypower_verifies_continuation": int(P6.get("pypower_verifies_continuation", pd.Series(dtype=bool)).fillna(False).sum())},
}
json.dump(summary, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=lambda o: None if (isinstance(o, float) and not np.isfinite(o)) else (o.item() if hasattr(o, "item") else str(o)))
print("VERDICT")
print(f"- Conventions: stored admittances match the raw parameters to {summary['conventions']['stored_vs_recomputed_admittance_max_rel_diff']:.0e}; the released solutions satisfy "
      f"PYPOWER's independently built equations to {summary['conventions']['released_mismatch_under_pypower_pu_max']:.0e} p.u.")
print(f"- Released cases: PYPOWER and our solver agree to {summary['released_cases']['pypower_vs_ours_max_dVm']:.0e} p.u.; overload sets identical in {agree_over.sum()} of {len(C2)} cases.")
print(f"- Unresolved constructed cases: PYPOWER resolved {summary['unresolved_n2_cases']['pypower_converged_any_start']} of {len(U3)} from classical starts.")
if REC.get("available") and REC.get("ours_from_genco_converged_30"):
    print(f"- Recovered case: a solution of the independent equations: {REC.get('pypower_from_our_state_converged')} (mismatch {REC.get('our_state_mismatch_under_pypower_pu', float('nan')):.0e} p.u.); "
          f"lowest voltage {REC.get('state_min_vm_all_buses', float('nan')):.3f} p.u., {REC.get('state_gens_beyond_q_limits')} limited PV buses beyond their reactive limits; "
          f"PYPOWER from classical starts (100 it.): {[REC[f'pypower_from_{n}_100']['converged'] for n in ['base', 'flat', 'dc']]}.")
elif REC.get("available"):
    print(f"- Recovered case: Newton from GENCO's answer did not converge in this run (30 it.: {REC.get('ours_from_genco_converged_30')}, 100 it.: {REC.get('ours_from_genco_converged_100')}); "
          f"PYPOWER from classical starts (100 it.): {[REC[f'pypower_from_{n}_100']['converged'] for n in ['base', 'flat', 'dc']]}.")
print(f"- Folds: continuation state is the highest-voltage solution for {int(FS.continuation_is_highest_voltage_solution.sum())} of {len(FS)} storms; "
      f"solvers converged beyond a fold at {int(FS.points_beyond_fold_where_any_solver_converged.sum())} points in all.")
print(f"- Reactive limits: see the table above (agreement with PYPOWER under our convention, differences under the MATPOWER convention).")
print(f"- Failed points: PYPOWER from GENCO's state converged at {summary['failed_points']['pypower_from_genco_100']} of {len(P6)} points (100 it.); ours at {summary['failed_points']['ours_from_genco_100']}.")
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"\nSaved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB). Send this file back.")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass